# 🧠 Human Person Detection: Deep Learning vs Traditional ML
## WMG9B7-15 — Artificial Intelligence & Deep Learning — Individual Assessment 2025/26

---

## 📋 README — How to Run This Notebook

**Environment:** Google Colab (GPU runtime recommended)

**Steps to run:**
1. Go to `Runtime → Change runtime type → T4 GPU`
2. Run **Section 1 (Setup)** first — this installs all dependencies
3. Run each section sequentially from top to bottom
4. The notebook is fully self-contained; all data is downloaded automatically

**Estimated total runtime:** ~60–90 minutes on Colab T4 GPU

**Models implemented:**
| # | Model | Type |
|---|-------|------|
| 1 | YOLOv8n (Pretrained Baseline) | Deep Learning |
| 2 | YOLOv8n (Fine-tuned on COCO person) | Deep Learning |
| 3 | Faster R-CNN (Pretrained + adapted) | Deep Learning |
| 4 | HOG + Linear SVM | Traditional ML |

**Dataset:** COCO 2017 (person class focus, subset for feasibility)

---


---
## Section 1: Environment Setup & Imports

We begin by installing all required libraries and importing them. We also set
global random seeds to ensure **full reproducibility** — any run of this notebook
will produce identical results.


In [ ]:
# ─── 1.1  Install dependencies ────────────────────────────────────────────────
# Run once at the start; Colab kernels persist installs for the session.

import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install("ultralytics>=8.2.0")          # YOLOv8 training + inference
pip_install("pycocotools")                 # COCO annotation parsing & evaluation
pip_install("scikit-image", "scikit-learn")# HOG feature extraction + SVM
pip_install("torchvision")                 # Faster R-CNN
pip_install("albumentations==1.3.1")       # Augmentation library compatible with this venv
pip_install("matplotlib", "seaborn")       # Visualisation
pip_install("tqdm")                        # Progress bars
pip_install("Pillow")                      # Image I/O

print("✅ All packages installed.")


In [ ]:
# ─── 1.2  Core imports ────────────────────────────────────────────────────────
import os, json, time, copy, random, shutil, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm

# ── PyTorch ──────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# ── COCO tools ───────────────────────────────────────────────────────────────
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# ── Classical ML ─────────────────────────────────────────────────────────────
from skimage.feature import hog
from skimage.transform import resize as sk_resize
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, precision_recall_fscore_support

# ── SciPy ────────────────────────────────────────────────────────────────────
from scipy.integrate import trapezoid

# ── YOLO ─────────────────────────────────────────────────────────────────────
from ultralytics import YOLO

warnings.filterwarnings("ignore")
print("✅ All imports successful.")

print(f"   PyTorch  : {torch.__version__}")
print(f"   Torchvision: {torchvision.__version__}")

In [ ]:
# ─── 1.3  Reproducibility & device setup ─────────────────────────────────────
# Setting seeds across ALL random number generators guarantees that every run
# of this notebook produces the exact same results — critical for academic work.

SEED = 42

def set_global_seed(seed: int):
    """Pin all RNG sources so every run is bit-for-bit identical."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic CUDA ops (slight speed cost, worth it for reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_global_seed(SEED)

# Detect available compute — falls back gracefully to CPU on machines without GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Seed set to {SEED}")
print(f"   Active device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU           : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ─── 1.4  Global path configuration ──────────────────────────────────────────
# All paths are defined here so they are easy to change without hunting through
# the notebook.  We use Pathlib for safe, OS-independent path handling.

BASE_DIR   = Path(r"C:\Users\u5756394\Desktop\u5756394\individual_assignment_DL\Human_detector\human_detection_project")
DATA_DIR   = BASE_DIR / "data"
COCO_DIR   = DATA_DIR / "coco"
YOLO_DIR   = BASE_DIR / "yolo_dataset"   # YOLO label format copy of COCO
MODEL_DIR  = BASE_DIR / "models"
RESULTS_DIR= BASE_DIR / "results"
VIZ_DIR    = RESULTS_DIR / "visualisations"

for d in [BASE_DIR, DATA_DIR, COCO_DIR, YOLO_DIR, MODEL_DIR, RESULTS_DIR, VIZ_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Dataset size constants (tune these to fit your Colab session) ─────────────
# COCO 2017 val contains 5,000 images; we use a reproducible subset for speed.
MAX_TRAIN_IMAGES = 3000   # images used for fine-tuning / Faster R-CNN training
MAX_VAL_IMAGES   = 500    # images used for validation / evaluation
MAX_TEST_IMAGES  = 200    # held-out test set for final comparison

# Detection thresholds
CONF_THRESHOLD   = 0.25   # minimum confidence to count as a detection
IOU_THRESHOLD    = 0.5    # IoU threshold for mAP@0.5

print("✅ Directory structure created.")
print(f"   Base : {BASE_DIR.resolve()}")


---
## Section 2: COCO Dataset Preparation

We use the **COCO 2017** dataset which is the standard benchmark for object detection.
COCO contains 80 object categories; we focus on **category 1 = "person"** but keep all
other objects in the images as background context — this is important because real
scenes are cluttered and a model must learn to distinguish people from other objects.

**Pipeline:**
1. Download COCO 2017 validation images + train images (subset)
2. Download annotations (JSON format)
3. Filter to person-relevant images
4. Build train / val / test splits
5. Export in YOLO format (for Models 1 & 2)
6. Build PyTorch `Dataset` objects (for Model 3)
7. Build HOG patch datasets (for Model 4)


In [ ]:
# ─── 2.1  Download COCO 2017 dataset ─────────────────────────────────────────
# Before downloading, we check whether files already exist locally.
# This prevents re-downloading on repeated runs (e.g. after a Colab restart).

import urllib.request, zipfile

def download_file(url: str, dest: Path, desc: str = ""):
    """Download a file with a progress indicator; skip if already present."""
    if dest.exists():
        print(f"   ✓ Already downloaded: {dest.name}  (skipping)")
        return
    print(f"   ↓ Downloading {desc or dest.name} …", end=" ", flush=True)
    t0 = time.time()
    urllib.request.urlretrieve(url, dest)
    elapsed = time.time() - t0
    size_mb = dest.stat().st_size / 1e6
    print(f"done ({size_mb:.0f} MB, {elapsed:.0f}s)")


def extract_zip(zip_path: Path, dest: Path, extracted_name: str):
    """Extract a zip archive; skip if the extracted folder already exists."""
    extracted_dir = dest / extracted_name
    if extracted_dir.exists():
        print(f"   ✓ Already extracted: {zip_path.name}  (skipping)")
        return
    print(f"   📦 Extracting {zip_path.name} …", end=" ", flush=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest)
    print("done")


COCO_BASE_URL = "http://images.cocodataset.org"
ANN_URL       = f"{COCO_BASE_URL}/annotations/annotations_trainval2017.zip"
TRAIN_IMG_URL = f"{COCO_BASE_URL}/zips/train2017.zip"

ann_zip   = DATA_DIR / "annotations_trainval2017.zip"
train_zip = DATA_DIR / "train2017.zip"

print("=== Downloading COCO 2017 (skipped if already present) ===")
download_file(ANN_URL,       ann_zip,   "annotations")
download_file(TRAIN_IMG_URL, train_zip, "train images (~18 GB)")

print("\n=== Extracting archives ===")
extract_zip(ann_zip,   COCO_DIR, "annotations")
extract_zip(train_zip, COCO_DIR, "train2017")

# Resolve annotation paths
ANN_TRAIN     = COCO_DIR / "annotations" / "instances_train2017.json"
ANN_VAL       = COCO_DIR / "annotations" / "instances_val2017.json"
TRAIN_IMG_DIR = COCO_DIR / "train2017"
VAL_IMG_DIR   = TRAIN_IMG_DIR   # alias used in later cells

print(f"\n✅ COCO data ready.")
print(f"   Train images : {TRAIN_IMG_DIR}")
print(f"   Annotations  : {COCO_DIR / 'annotations'}")


In [ ]:
# ─── 2.2  Parse COCO annotations & filter to person images ──────────────────
# COCO uses a JSON annotation format.  We use pycocotools (the official API)
# to parse it efficiently.  We then identify images that contain at least one
# "person" bounding box with area > 1000 px² (to exclude tiny, hard-to-see
# instances that would add noise to training).

PERSON_CAT_ID = 1   # COCO person category ID is always 1
MIN_AREA      = 1000  # minimum bounding box area in pixels²

print("Loading COCO train annotations …")
coco_val = COCO(str(ANN_TRAIN))  # reused downstream name to minimise notebook changes

# Get all image IDs that have at least one person annotation
person_ann_ids = coco_val.getAnnIds(catIds=[PERSON_CAT_ID])
person_anns    = coco_val.loadAnns(person_ann_ids)

# Filter by minimum area — skip tiny, barely-visible people
valid_anns     = [a for a in person_anns if a["area"] >= MIN_AREA]
person_img_ids = list(set(a["image_id"] for a in valid_anns))

print(f"   Total person annotations  : {len(person_anns):,}")
print(f"   After area filter (≥{MIN_AREA}px²): {len(valid_anns):,}")
print(f"   Unique images with persons: {len(person_img_ids):,}")


In [ ]:
MAX_TRAIN_IMAGES = 50000
MAX_VAL_IMAGES = 10000
MAX_TEST_IMAGES = 10000

In [ ]:
# ─── 2.3  Create reproducible train / val / test splits ──────────────────────
# We use the COCO train2017 split as our full image pool.  We then create 3
# sub-splits:
#   • train  : used for fine-tuning YOLO and training Faster R-CNN + SVM
#   • val    : used during training for early stopping / hyperparameter tuning
#   • test   : completely held-out; used only for final evaluation & comparison
#
# The random shuffle uses SEED so splits are reproducible.

random.seed(SEED)
shuffled_ids = person_img_ids.copy()
random.shuffle(shuffled_ids)

# Limit pool size to what's available
desired_total = MAX_TRAIN_IMAGES + MAX_VAL_IMAGES + MAX_TEST_IMAGES
pool = shuffled_ids[:min(len(shuffled_ids), desired_total)]
pool_size = len(pool)

# If the filtered pool is smaller than requested, fall back to proportional
# splits so validation and test sets are never empty when we have enough data.
if pool_size < desired_total:
    print(f"⚠ Requested {desired_total:,} images, but only {pool_size:,} are available after filtering.")
    if pool_size >= 3:
        n_train = max(1, int(round(pool_size * 0.8)))
        n_val   = max(1, int(round(pool_size * 0.1)))
        n_test  = pool_size - n_train - n_val
        if n_test <= 0:
            n_test = 1
            n_train = max(1, pool_size - n_val - n_test)
    else:
        n_train = pool_size
        n_val   = 0
        n_test  = 0
else:
    n_train = MAX_TRAIN_IMAGES
    n_val   = MAX_VAL_IMAGES
    n_test  = MAX_TEST_IMAGES

# Clamp counts to the pool size in case of rounding
n_train = min(n_train, pool_size)
n_val   = min(n_val, max(0, pool_size - n_train))
n_test  = max(0, pool_size - n_train - n_val)

train_ids = pool[:n_train]
val_ids   = pool[n_train:n_train + n_val]
test_ids  = pool[n_train + n_val:n_train + n_val + n_test]

print(f"✅ Dataset splits (COCO train2017 person images):")
print(f"   Train : {len(train_ids):>5,} images")
print(f"   Val   : {len(val_ids):>5,} images")
print(f"   Test  : {len(test_ids):>5,} images")
print(f"   Total : {len(train_ids)+len(val_ids)+len(test_ids):>5,} images")

# Save split IDs for reference / reproducibility
splits = {"train": train_ids, "val": val_ids, "test": test_ids, "seed": SEED}
with open(RESULTS_DIR / "dataset_splits.json", "w") as f:
    json.dump(splits, f)
print(f"   Splits saved to {RESULTS_DIR / 'dataset_splits.json'}")


In [ ]:
# ─── 2.4  Quick dataset visualisation ────────────────────────────────────────
# Before building models, it is good practice to inspect the data.
# We visualise a sample of training images with their ground-truth bounding boxes.

def draw_coco_boxes(ax, img_id: int, coco_api: COCO, img_dir: Path,
                    max_boxes: int = 10, title: str = ""):
    """Draw a COCO image with ground-truth person bounding boxes."""
    img_info = coco_api.loadImgs(img_id)[0]
    img_path = img_dir / img_info["file_name"]

    if not img_path.exists():
        ax.axis("off")
        ax.set_title("Image not found", fontsize=8)
        return

    img = np.array(Image.open(img_path).convert("RGB"))
    ax.imshow(img)

    ann_ids = coco_api.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
    anns    = coco_api.loadAnns(ann_ids)[:max_boxes]

    for ann in anns:
        x, y, w, h = ann["bbox"]
        rect = mpatches.Rectangle((x, y), w, h,
                                   linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)

    ax.set_title(f"{title}\n{img_info['file_name']} | {len(anns)} persons",
                 fontsize=7)
    ax.axis("off")


# Show 6 random training images
sample_ids = random.sample(train_ids, 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Sample COCO Training Images — Ground Truth Person Boxes (green)",
             fontsize=13, fontweight="bold")

for ax, img_id in zip(axes.flat, sample_ids):
    draw_coco_boxes(ax, img_id, coco_val, VAL_IMG_DIR)

plt.tight_layout()
plt.savefig(VIZ_DIR / "00_dataset_samples.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Dataset sample saved.")


In [ ]:
# ─── 2.5  Export COCO annotations to YOLO label format ───────────────────────
# YOLOv8 expects labels in its own text-file format:
#   <class_id> <x_centre_norm> <y_centre_norm> <width_norm> <height_norm>
# All values normalised to [0, 1] relative to image width/height.
#
# We set class_id = 0 for "person" (single-class setup) so YOLO focuses only
# on people.  Other objects in the image provide implicit negative context.

def coco_bbox_to_yolo(bbox, img_w, img_h):
    """Convert COCO [x,y,w,h] (pixel) → YOLO [xc,yc,w,h] (normalised)."""
    x, y, w, h = bbox
    xc = (x + w / 2) / img_w
    yc = (y + h / 2) / img_h
    wn = w / img_w
    hn = h / img_h
    return xc, yc, wn, hn


def export_yolo_split(img_ids, coco_api, img_dir, split_name):
    """Write images (symlinks) and YOLO label .txt files for one split."""
    img_out = YOLO_DIR / split_name / "images"
    lbl_out = YOLO_DIR / split_name / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    exported, skipped = 0, 0
    for img_id in tqdm(img_ids, desc=f"  {split_name}", leave=False):
        img_info = coco_api.loadImgs(img_id)[0]
        src_path = img_dir / img_info["file_name"]
        if not src_path.exists():
            skipped += 1
            continue

        dst_img  = img_out / img_info["file_name"]
        dst_lbl  = lbl_out / (Path(img_info["file_name"]).stem + ".txt")

        # Copy image (use hard link to save disk space)
        if not dst_img.exists():
            shutil.copy2(src_path, dst_img)

        # Build label file
        ann_ids = coco_api.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        anns    = coco_api.loadAnns(ann_ids)

        lines = []
        for ann in anns:
            if ann["area"] < MIN_AREA:
                continue
            xc, yc, wn, hn = coco_bbox_to_yolo(
                ann["bbox"], img_info["width"], img_info["height"])
            # Clamp to [0,1] — occasionally COCO boxes slightly exceed image bounds
            xc = max(0.0, min(1.0, xc))
            yc = max(0.0, min(1.0, yc))
            wn = max(0.001, min(1.0, wn))
            hn = max(0.001, min(1.0, hn))
            lines.append(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")

        if lines:   # Only write label file if there are valid annotations
            with open(dst_lbl, "w") as f:
                f.write("\n".join(lines))
            exported += 1

    return exported, skipped


print("=== Exporting YOLO-format dataset ===")
for split_name, ids in [("train", train_ids), ("val", val_ids), ("test", test_ids)]:
    exp, skp = export_yolo_split(ids, coco_val, VAL_IMG_DIR, split_name)
    print(f"   {split_name:5s}: {exp:,} images exported, {skp} skipped")

print("✅ YOLO dataset export complete.")


In [ ]:
# ─── 2.6  Write YOLO dataset YAML configuration ──────────────────────────────
# Ultralytics reads a YAML file that tells it where the data lives and what
# the class names are.  This file is required for both training and evaluation.

yolo_yaml_content = f"""# YOLO dataset configuration — Person detection on COCO subset
# Auto-generated by human_detection_project notebook

path: {YOLO_DIR.resolve()}   # dataset root directory
train: train/images           # relative to path
val:   val/images             # relative to path
test:  test/images            # relative to path

nc: 1                         # number of classes (person only)
names:
  0: person
"""

yolo_yaml_path = YOLO_DIR / "person_detection.yaml"
with open(yolo_yaml_path, "w") as f:
    f.write(yolo_yaml_content)

print(f"✅ YOLO YAML written to: {yolo_yaml_path}")
print()
print(yolo_yaml_content)


In [ ]:
# ─── 2.7  Build PyTorch Dataset for Faster R-CNN ─────────────────────────────
# torchvision's detection models expect a specific input format:
#   image  : FloatTensor [3, H, W] in [0, 1]
#   target : dict with 'boxes' (FloatTensor [N,4]) and 'labels' (Int64Tensor [N])
#
# We use XYXY box format (required by torchvision detectors).

class COCOPersonDataset(Dataset):
    """
    PyTorch Dataset that loads COCO images and returns person bounding boxes
    in the format expected by torchvision Faster R-CNN.

    Args:
        img_ids   : list of COCO image IDs to include
        coco_api  : initialised pycocotools COCO object
        img_dir   : directory containing COCO images
        transforms: optional callable applied to (image, target)
        min_area  : discard boxes smaller than this area (px²)
    """

    def __init__(self, img_ids, coco_api, img_dir, transforms=None,
                 min_area=MIN_AREA):
        self.img_ids    = img_ids
        self.coco       = coco_api
        self.img_dir    = Path(img_dir)
        self.transforms = transforms
        self.min_area   = min_area

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id   = self.img_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = self.img_dir / img_info["file_name"]

        # Load image as float tensor in [0, 1]
        img = Image.open(img_path).convert("RGB")
        img = T.ToTensor()(img)   # shape [3, H, W]

        # Load annotations, filter tiny boxes
        ann_ids = self.coco.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        anns    = self.coco.loadAnns(ann_ids)
        anns    = [a for a in anns if a["area"] >= self.min_area]

        if not anns:
            # Negative image (no person boxes after filtering)
            # Return a dummy target so the dataloader doesn't crash
            target = {
                "boxes":    torch.zeros((0, 4), dtype=torch.float32),
                "labels":   torch.zeros((0,),   dtype=torch.int64),
                "image_id": torch.tensor([img_id]),
            }
        else:
            # Convert COCO [x, y, w, h] → XYXY format for torchvision
            boxes = []
            for a in anns:
                x, y, w, h = a["bbox"]
                boxes.append([x, y, x + w, y + h])
            boxes  = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.ones(len(boxes), dtype=torch.int64)  # class 1 = person

            target = {
                "boxes":    boxes,
                "labels":   labels,
                "image_id": torch.tensor([img_id]),
            }

        if self.transforms:
            img, target = self.transforms(img, target)

        return img, target


def collate_fn(batch):
    """Custom collate because images can have different sizes."""
    return tuple(zip(*batch))


# Instantiate datasets — we'll use these in Section 5 (Faster R-CNN)
train_dataset = COCOPersonDataset(train_ids, coco_val, VAL_IMG_DIR)
val_dataset   = COCOPersonDataset(val_ids,   coco_val, VAL_IMG_DIR)
test_dataset  = COCOPersonDataset(test_ids,  coco_val, VAL_IMG_DIR)

print(f"✅ PyTorch Datasets ready:")
print(f"   Train : {len(train_dataset):,} images")
print(f"   Val   : {len(val_dataset):,} images")
print(f"   Test  : {len(test_dataset):,} images")


---
## Section 2.8 — Exploratory Data Analysis (EDA)

Before training any model we **thoroughly analyse the dataset** to:
- Understand image-level statistics (resolution, aspect ratio)
- Quantify the class / annotation distribution
- Study bounding-box size and shape distributions
- Identify potential data-quality issues (tiny boxes, crowd annotations)
- Derive model-specific insights for YOLO, Faster R-CNN, and HOG+SVM

Each sub-section targets one specific question and ends with an
**interpretation** cell that summarises the finding.


In [ ]:
# ─── 2.8.1  Global image statistics ──────────────────────────────────────────
# We analyse the raw COCO images in our train split to understand the
# resolution landscape — this drives decisions like imgsz=640 for YOLO.

print("=" * 65)
print("  EDA 2.8.1 — Image Resolution & Aspect Ratio (Train Split)")
print("=" * 65)

widths, heights, aspect_ratios = [], [], []

for img_id in tqdm(train_ids, desc="  Scanning images", leave=False):
    info = coco_val.loadImgs(img_id)[0]
    w, h = info["width"], info["height"]
    widths.append(w); heights.append(h)
    aspect_ratios.append(w / h)

widths  = np.array(widths)
heights = np.array(heights)
aspect_ratios = np.array(aspect_ratios)

print(f"  Train images: {len(widths):,}")
print(f"  Width  — min: {widths.min()}, max: {widths.max()}, "
      f"mean: {widths.mean():.0f}, median: {np.median(widths):.0f}")
print(f"  Height — min: {heights.min()}, max: {heights.max()}, "
      f"mean: {heights.mean():.0f}, median: {np.median(heights):.0f}")
print(f"  Aspect — min: {aspect_ratios.min():.2f}, max: {aspect_ratios.max():.2f}, "
      f"mean: {aspect_ratios.mean():.2f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("EDA 2.8.1 — Image Resolution Distribution (Train Split)",
             fontsize=13, fontweight="bold")

axes[0].hist(widths,  bins=40, color="#4C72B0", edgecolor="white", linewidth=0.5)
axes[0].axvline(widths.mean(),  color="red",    linestyle="--", label=f"Mean {widths.mean():.0f}")
axes[0].axvline(np.median(widths), color="orange", linestyle=":", label=f"Median {np.median(widths):.0f}")
axes[0].set_title("Image Width (px)"); axes[0].set_xlabel("px"); axes[0].legend(fontsize=8)

axes[1].hist(heights, bins=40, color="#55A868", edgecolor="white", linewidth=0.5)
axes[1].axvline(heights.mean(),  color="red",    linestyle="--", label=f"Mean {heights.mean():.0f}")
axes[1].axvline(np.median(heights), color="orange", linestyle=":", label=f"Median {np.median(heights):.0f}")
axes[1].set_title("Image Height (px)"); axes[1].set_xlabel("px"); axes[1].legend(fontsize=8)

axes[2].hist(aspect_ratios, bins=40, color="#C44E52", edgecolor="white", linewidth=0.5)
axes[2].axvline(1.0, color="black", linestyle="-", linewidth=1, label="Square (1:1)")
axes[2].set_title("Aspect Ratio (W/H)"); axes[2].set_xlabel("W/H ratio"); axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(VIZ_DIR / "EDA_01_image_resolution.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ EDA 2.8.1 complete.")


In [ ]:
# ─── 2.8.2  Annotation distribution ─────────────────────────────────────────
# How many person instances per image?  Are images crowded or sparse?
# A bimodal distribution would suggest different scene types.

print("=" * 65)
print("  EDA 2.8.2 — Person Annotations Per Image")
print("=" * 65)

persons_per_img = []
for img_id in tqdm(train_ids, desc="  Counting annotations", leave=False):
    ann_ids = coco_val.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
    anns    = [a for a in coco_val.loadAnns(ann_ids) if a["area"] >= MIN_AREA]
    persons_per_img.append(len(anns))

persons_per_img = np.array(persons_per_img)

print(f"  Images with ≥1 person  : {(persons_per_img > 0).sum():,}  "
      f"({(persons_per_img > 0).mean()*100:.1f}%)")
print(f"  Persons/image — mean   : {persons_per_img.mean():.2f}")
print(f"  Persons/image — median : {np.median(persons_per_img):.1f}")
print(f"  Persons/image — max    : {persons_per_img.max()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("EDA 2.8.2 — Person Count Distribution", fontsize=13, fontweight="bold")

axes[0].hist(persons_per_img, bins=range(0, persons_per_img.max() + 2),
             color="#4C72B0", edgecolor="white", linewidth=0.5, align="left")
axes[0].set_title("Persons per Image (full range)")
axes[0].set_xlabel("# persons"); axes[0].set_ylabel("# images")
axes[0].axvline(persons_per_img.mean(), color="red", linestyle="--",
                label=f"Mean {persons_per_img.mean():.1f}")
axes[0].legend(fontsize=9)

# Zoomed view (≤ 20 persons)
clipped = persons_per_img[persons_per_img <= 20]
axes[1].hist(clipped, bins=range(0, 22), color="#55A868", edgecolor="white",
             linewidth=0.5, align="left")
axes[1].set_title("Persons per Image (zoomed ≤20)")
axes[1].set_xlabel("# persons"); axes[1].set_ylabel("# images")

plt.tight_layout()
plt.savefig(VIZ_DIR / "EDA_02_persons_per_image.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ EDA 2.8.2 complete.")


In [ ]:
# ─── 2.8.3  Bounding box size & shape analysis ───────────────────────────────
# Understanding bbox sizes drives anchor design (Faster R-CNN / YOLO) and
# informs HOG window size selection.  We analyse width, height, area, and
# relative size (box area / image area).

print("=" * 65)
print("  EDA 2.8.3 — Bounding Box Size & Shape")
print("=" * 65)

box_widths, box_heights, box_areas, box_rel_areas, box_aspects = [], [], [], [], []
crowd_count, small_count = 0, 0

for img_id in tqdm(train_ids, desc="  Analysing boxes", leave=False):
    info    = coco_val.loadImgs(img_id)[0]
    img_w, img_h = info["width"], info["height"]
    ann_ids = coco_val.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
    for ann in coco_val.loadAnns(ann_ids):
        if ann.get("iscrowd", 0):
            crowd_count += 1; continue
        x, y, w, h = ann["bbox"]
        area = ann["area"]
        if area < MIN_AREA:
            small_count += 1; continue
        box_widths.append(w); box_heights.append(h)
        box_areas.append(area)
        box_rel_areas.append(area / (img_w * img_h))
        box_aspects.append(w / max(h, 1))

box_widths    = np.array(box_widths)
box_heights   = np.array(box_heights)
box_areas     = np.array(box_areas)
box_rel_areas = np.array(box_rel_areas)
box_aspects   = np.array(box_aspects)

print(f"  Valid boxes    : {len(box_areas):,}")
print(f"  Crowd boxes    : {crowd_count:,}  (excluded)")
print(f"  Tiny boxes (<{MIN_AREA}px²): {small_count:,}  (excluded)")
print(f"  Width   mean/median: {box_widths.mean():.0f} / {np.median(box_widths):.0f} px")
print(f"  Height  mean/median: {box_heights.mean():.0f} / {np.median(box_heights):.0f} px")
print(f"  Rel area mean:  {box_rel_areas.mean()*100:.2f}% of image")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("EDA 2.8.3 — Bounding Box Statistics (Train Split)",
             fontsize=13, fontweight="bold")

axes[0,0].hist(box_widths,  bins=50, color="#4C72B0", edgecolor="white", lw=0.5)
axes[0,0].set_title("Box Width (px)"); axes[0,0].set_xlabel("px")
axes[0,0].axvline(box_widths.mean(),     color="red",    ls="--", label=f"Mean {box_widths.mean():.0f}")
axes[0,0].axvline(np.median(box_widths), color="orange", ls=":",  label=f"Median {np.median(box_widths):.0f}")
axes[0,0].legend(fontsize=8)

axes[0,1].hist(box_heights, bins=50, color="#55A868", edgecolor="white", lw=0.5)
axes[0,1].set_title("Box Height (px)"); axes[0,1].set_xlabel("px")
axes[0,1].axvline(box_heights.mean(),     color="red",    ls="--", label=f"Mean {box_heights.mean():.0f}")
axes[0,1].axvline(np.median(box_heights), color="orange", ls=":",  label=f"Median {np.median(box_heights):.0f}")
axes[0,1].legend(fontsize=8)

axes[0,2].hist(np.log10(box_areas + 1), bins=50, color="#C44E52", edgecolor="white", lw=0.5)
axes[0,2].set_title("Box Area (log₁₀ scale)"); axes[0,2].set_xlabel("log₁₀(area px²)")
axes[0,2].axvline(np.log10(MIN_AREA), color="black", ls="--", lw=1.5,
                  label=f"Min area filter ({MIN_AREA}px²)")
axes[0,2].legend(fontsize=8)

axes[1,0].hist(box_aspects, bins=50, color="#8172B2", edgecolor="white", lw=0.5)
axes[1,0].set_title("Box Aspect Ratio (W/H)"); axes[1,0].set_xlabel("W/H")
axes[1,0].axvline(1.0, color="black", ls="-", lw=1, label="Square")
axes[1,0].legend(fontsize=8)

axes[1,1].hist(box_rel_areas * 100, bins=50, color="#CCB974", edgecolor="white", lw=0.5)
axes[1,1].set_title("Box Relative Area (% of image)"); axes[1,1].set_xlabel("% of image area")

# Scatter: width vs height (sampled for speed)
sample_n = min(5000, len(box_widths))
idx = np.random.choice(len(box_widths), sample_n, replace=False)
axes[1,2].scatter(box_widths[idx], box_heights[idx],
                  alpha=0.15, s=6, c="#4C72B0")
axes[1,2].set_title(f"Box Width vs Height (n={sample_n:,})")
axes[1,2].set_xlabel("Width (px)"); axes[1,2].set_ylabel("Height (px)")
# Reference lines: PATCH_SIZE used by HOG
# axes[1,2].axhline(PATCH_SIZE[0], color="red",   ls="--", lw=1.5, label=f"HOG H={PATCH_SIZE[0]}")
# axes[1,2].axvline(PATCH_SIZE[1], color="orange", ls="--", lw=1.5, label=f"HOG W={PATCH_SIZE[1]}")
axes[1,2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(VIZ_DIR / "EDA_03_bbox_statistics.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ EDA 2.8.3 complete.")


In [ ]:
# ─── 2.8.4  Split balance & class balance ────────────────────────────────────
# Verifies that train / val / test splits are balanced in terms of:
#   • Number of images
#   • Total person instances
#   • Average persons per image
# An imbalanced split would introduce evaluation bias.

print("=" * 65)
print("  EDA 2.8.4 — Split Balance Analysis")
print("=" * 65)

split_stats = {}
for split_name, ids in [("Train", train_ids), ("Val", val_ids), ("Test", test_ids)]:
    counts = []
    for img_id in ids:
        ann_ids = coco_val.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        anns    = [a for a in coco_val.loadAnns(ann_ids) if a["area"] >= MIN_AREA]
        counts.append(len(anns))
    counts = np.array(counts)
    split_stats[split_name] = {
        "n_images":    len(ids),
        "n_persons":   counts.sum(),
        "mean_per_img": counts.mean(),
        "median_per_img": float(np.median(counts)),
    }
    print(f"  {split_name:5s}: {len(ids):>5,} images | "
          f"{counts.sum():>6,} persons | mean {counts.mean():.2f}/img")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("EDA 2.8.4 — Split Balance", fontsize=13, fontweight="bold")

names  = list(split_stats.keys())
colors = ["#4C72B0", "#55A868", "#C44E52"]

axes[0].bar(names, [split_stats[n]["n_images"]   for n in names], color=colors)
axes[0].set_title("Images per Split"); axes[0].set_ylabel("# images")
for i, n in enumerate(names):
    axes[0].text(i, split_stats[n]["n_images"] * 0.5,
                 f"{split_stats[n]['n_images']:,}", ha="center", color="white", fontweight="bold")

axes[1].bar(names, [split_stats[n]["n_persons"]  for n in names], color=colors)
axes[1].set_title("Person Instances per Split"); axes[1].set_ylabel("# instances")
for i, n in enumerate(names):
    axes[1].text(i, split_stats[n]["n_persons"] * 0.5,
                 f"{split_stats[n]['n_persons']:,}", ha="center", color="white", fontweight="bold")

axes[2].bar(names, [split_stats[n]["mean_per_img"] for n in names], color=colors)
axes[2].set_title("Avg Persons per Image"); axes[2].set_ylabel("mean count")
for i, n in enumerate(names):
    axes[2].text(i, split_stats[n]["mean_per_img"] * 0.5,
                 f"{split_stats[n]['mean_per_img']:.2f}", ha="center", color="white", fontweight="bold")

plt.tight_layout()
plt.savefig(VIZ_DIR / "EDA_04_split_balance.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ EDA 2.8.4 complete.")


In [ ]:
# ─── 2.8.5  YOLO-specific EDA ────────────────────────────────────────────────
# YOLOv8 uses normalised [xc, yc, w, h] labels.  We verify the distribution
# of these values in our exported YOLO dataset, checking:
#   • Spatial distribution of box centres (should cover the full image)
#   • Normalised width/height distribution (drives anchor generation)
#   • Multi-scale coverage (how many tiny / medium / large boxes)
#
# 🔎 Relevance: YOLO's anchor-free detection head still benefits from
#    understanding the scale distribution to set imgsz and augmentation.

print("=" * 65)
print("  EDA 2.8.5 — YOLO Label Distribution")
print("=" * 65)

yolo_train_lbl = YOLO_DIR / "train" / "labels"
xc_vals, yc_vals, wn_vals, hn_vals = [], [], [], []

label_files = list(yolo_train_lbl.glob("*.txt"))
print(f"  Label files found: {len(label_files):,}")

for lf in tqdm(label_files[:3000], desc="  Parsing YOLO labels", leave=False):
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                _, xc, yc, wn, hn = map(float, parts)
                xc_vals.append(xc); yc_vals.append(yc)
                wn_vals.append(wn); hn_vals.append(hn)

xc_vals = np.array(xc_vals); yc_vals = np.array(yc_vals)
wn_vals = np.array(wn_vals); hn_vals = np.array(hn_vals)
print(f"  Total labels parsed: {len(xc_vals):,}")

# Scale categories
tiny   = (wn_vals < 0.05) | (hn_vals < 0.05)
large  = (wn_vals > 0.5)  | (hn_vals > 0.5)
medium = ~tiny & ~large
print(f"  Tiny boxes   (w or h < 5%): {tiny.sum():,}  ({tiny.mean()*100:.1f}%)")
print(f"  Medium boxes              : {medium.sum():,}  ({medium.mean()*100:.1f}%)")
print(f"  Large boxes  (w or h > 50%): {large.sum():,}  ({large.mean()*100:.1f}%)")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("EDA 2.8.5 — YOLO Label Statistics (Normalised Coordinates)",
             fontsize=13, fontweight="bold")

axes[0,0].hist2d(xc_vals, yc_vals, bins=50, cmap="YlOrRd")
axes[0,0].set_title("Box Centre Heatmap (xc, yc)")
axes[0,0].set_xlabel("xc (normalised)"); axes[0,0].set_ylabel("yc (normalised)")
axes[0,0].invert_yaxis()  # image coordinates: y increases downward

axes[0,1].hist(wn_vals, bins=50, color="#4C72B0", edgecolor="white", lw=0.5)
axes[0,1].set_title("Normalised Box Width"); axes[0,1].set_xlabel("wn")
axes[0,1].axvline(wn_vals.mean(), color="red", ls="--", label=f"Mean {wn_vals.mean():.3f}")
axes[0,1].legend(fontsize=8)

axes[0,2].hist(hn_vals, bins=50, color="#55A868", edgecolor="white", lw=0.5)
axes[0,2].set_title("Normalised Box Height"); axes[0,2].set_xlabel("hn")
axes[0,2].axvline(hn_vals.mean(), color="red", ls="--", label=f"Mean {hn_vals.mean():.3f}")
axes[0,2].legend(fontsize=8)

# Width vs Height scatter
sample_n = min(5000, len(wn_vals))
idx = np.random.choice(len(wn_vals), sample_n, replace=False)
axes[1,0].scatter(wn_vals[idx], hn_vals[idx], alpha=0.2, s=5, c="#4C72B0")
axes[1,0].set_title(f"Box W vs H (normalised, n={sample_n:,})")
axes[1,0].set_xlabel("wn"); axes[1,0].set_ylabel("hn")

# Scale category pie
axes[1,1].pie([tiny.sum(), medium.sum(), large.sum()],
              labels=["Tiny", "Medium", "Large"],
              colors=["#C44E52", "#4C72B0", "#55A868"],
              autopct="%1.1f%%", startangle=140, textprops={"fontsize": 10})
axes[1,1].set_title("Box Scale Distribution")

# Aspect ratio (wn/hn)
ar = wn_vals / np.maximum(hn_vals, 1e-6)
axes[1,2].hist(ar, bins=50, color="#8172B2", edgecolor="white", lw=0.5)
axes[1,2].axvline(1.0, color="black", ls="-", lw=1.5, label="Square")
axes[1,2].set_title("Aspect Ratio (wn/hn)"); axes[1,2].set_xlabel("wn/hn")
axes[1,2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(VIZ_DIR / "EDA_05_yolo_labels.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ EDA 2.8.5 (YOLO) complete.")


In [ ]:
# ─── 2.8.6  Faster R-CNN–specific EDA ────────────────────────────────────────
# Faster R-CNN uses a fixed set of anchors at three scales and three ratios.
# We analyse the actual GT box statistics to check how well the default
# anchors (32,64,128,256,512 × ratios 0.5,1,2) match our data.
#
# 🔎 Relevance: Anchor mismatch → low recall for extreme-aspect-ratio boxes.

print("=" * 65)
print("  EDA 2.8.6 — Faster R-CNN Anchor Coverage Analysis")
print("=" * 65)

# Default torchvision Faster R-CNN anchor sizes and ratios
ANCHOR_SIZES  = [32, 64, 128, 256, 512]
ANCHOR_RATIOS = [0.5, 1.0, 2.0]

all_gt_w, all_gt_h = [], []
for img_id in tqdm(train_ids[:1000], desc="  Scanning GT boxes", leave=False):
    info = coco_val.loadImgs(img_id)[0]
    ann_ids = coco_val.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
    for ann in coco_val.loadAnns(ann_ids):
        if ann["area"] >= MIN_AREA and not ann.get("iscrowd", 0):
            _, _, w, h = ann["bbox"]
            all_gt_w.append(w); all_gt_h.append(h)

all_gt_w = np.array(all_gt_w)
all_gt_h = np.array(all_gt_h)
gt_areas = np.sqrt(all_gt_w * all_gt_h)  # "effective scale" = sqrt(area)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("EDA 2.8.6 — Faster R-CNN: GT Box Scale vs Default Anchors",
             fontsize=13, fontweight="bold")

# Scale histogram with anchor lines
axes[0].hist(gt_areas, bins=60, color="#4C72B0", edgecolor="white", lw=0.5, label="GT √area")
for s in ANCHOR_SIZES:
    axes[0].axvline(s, color="red", ls="--", lw=1.2, alpha=0.8, label=f"Anchor {s}")
axes[0].set_title("GT √(area) vs Anchor Scales")
axes[0].set_xlabel("√(w×h) px"); axes[0].set_ylabel("# boxes")
axes[0].legend(fontsize=7, ncol=2)

# Aspect ratio
gt_ratio = all_gt_w / np.maximum(all_gt_h, 1e-6)
axes[1].hist(gt_ratio, bins=60, color="#55A868", edgecolor="white", lw=0.5, label="GT W/H")
for r in ANCHOR_RATIOS:
    axes[1].axvline(r, color="red", ls="--", lw=1.2, alpha=0.8, label=f"Ratio {r}")
axes[1].set_title("GT Aspect Ratio vs Anchor Ratios")
axes[1].set_xlabel("W/H"); axes[1].legend(fontsize=8)

# 2D hex density: sqrt(area) vs aspect ratio
axes[2].hexbin(gt_areas, gt_ratio, gridsize=30, cmap="Blues", mincnt=1)
axes[2].set_xlabel("√(w×h)"); axes[2].set_ylabel("W/H")
axes[2].set_title("Scale × Aspect Ratio Joint Density")

pct_covered = ((gt_areas >= ANCHOR_SIZES[0]) & (gt_areas <= ANCHOR_SIZES[-1])).mean() * 100
print(f"  GT boxes within anchor scale range "
      f"[{ANCHOR_SIZES[0]}–{ANCHOR_SIZES[-1]}]: {pct_covered:.1f}%")

plt.tight_layout()
plt.savefig(VIZ_DIR / "EDA_06_frcnn_anchor_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ EDA 2.8.6 (Faster R-CNN) complete.")


In [ ]:
# ─── 2.8.7  HOG+SVM–specific EDA ─────────────────────────────────────────────
# HOG relies on correctly-sized patches.  We analyse:
#   • Fraction of person boxes that fit the HOG window (128×64)
#   • HOG feature statistics across a random sample of positive patches
#   • Mean person patch appearance (qualitative visual check)
#
# 🔎 Relevance: Boxes smaller than PATCH_SIZE cannot be directly used
#    as positive training samples and must be upscaled — this risks
#    aliasing artefacts.

print("=" * 65)
print("  EDA 2.8.7 — HOG + SVM Patch Statistics")
print("=" * 65)

ph, pw = PATCH_SIZE  # 128, 64
fits_window  = (box_heights >= ph) & (box_widths >= pw)
needs_upsamp = ~fits_window

print(f"  Person boxes ≥ HOG window ({pw}×{ph}): "
      f"{fits_window.sum():,}  ({fits_window.mean()*100:.1f}%)")
print(f"  Need upsampling (too small)         : "
      f"{needs_upsamp.sum():,}  ({needs_upsamp.mean()*100:.1f}%)")

# Sample positive patches and compute mean appearance
print("  Sampling positive patches for mean appearance …")
sample_crops = []
rng = np.random.default_rng(SEED)
sampled_ids  = rng.choice(train_ids, size=min(200, len(train_ids)), replace=False)

for img_id in tqdm(sampled_ids, desc="  Sampling", leave=False):
    if len(sample_crops) >= 100: break
    info    = coco_val.loadImgs(img_id)[0]
    img_np  = np.array(Image.open(VAL_IMG_DIR / info["file_name"]).convert("RGB"))
    ann_ids = coco_val.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
    for ann in coco_val.loadAnns(ann_ids):
        if ann["area"] < MIN_AREA or ann.get("iscrowd", 0): continue
        x, y, w, h = map(int, ann["bbox"])
        crop = img_np[max(0,y):y+h, max(0,x):x+w]
        if crop.shape[0] > 0 and crop.shape[1] > 0:
            from skimage.transform import resize as sk_resize_local
            resized = (sk_resize_local(crop, PATCH_SIZE, anti_aliasing=True) * 255).astype(np.uint8)
            sample_crops.append(resized)
            break

mean_patch = np.mean(sample_crops, axis=0).astype(np.uint8) if sample_crops else None

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("EDA 2.8.7 — HOG + SVM Patch Analysis", fontsize=13, fontweight="bold")

# Size suitability
labels = [f"Fits ({fits_window.mean()*100:.0f}%)", f"Too small ({needs_upsamp.mean()*100:.0f}%)"]
axes[0].pie([fits_window.sum(), needs_upsamp.sum()],
            labels=labels, colors=["#55A868", "#C44E52"],
            autopct="%1.1f%%", startangle=90, textprops={"fontsize": 10})
axes[0].set_title(f"Patch Size Fit for HOG Window ({pw}×{ph})")

# Mean positive patch
if mean_patch is not None:
    axes[1].imshow(mean_patch)
    axes[1].set_title(f"Mean Person Patch (n={len(sample_crops)})")
    axes[1].axis("off")
else:
    axes[1].text(0.5, 0.5, "No patches found", ha="center")

# HOG feature histogram (one sample)
if sample_crops:
    sample_for_hog = sample_crops[0]
    grey_sample = (0.2126*sample_for_hog[...,0] +
                   0.7152*sample_for_hog[...,1] +
                   0.0722*sample_for_hog[...,2])
    feat = hog(grey_sample / 255.0, **HOG_PARAMS)
    axes[2].hist(feat, bins=50, color="#4C72B0", edgecolor="white", lw=0.5)
    axes[2].set_title(f"HOG Feature Distribution (1 patch, dim={len(feat)})")
    axes[2].set_xlabel("HOG value")

plt.tight_layout()
plt.savefig(VIZ_DIR / "EDA_07_hog_patch_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ EDA 2.8.7 (HOG+SVM) complete.  Sampled {len(sample_crops)} patches.")


In [ ]:
# ─── 2.8.8  EDA Summary ───────────────────────────────────────────────────────
print("=" * 65)
print("  EDA Summary — Key Findings & Model Implications")
print("=" * 65)
print()
print("  Image statistics:")
print(f"    Median resolution : {int(np.median(widths))}×{int(np.median(heights))} px")
print(f"    Aspect ratio      : most images are landscape (mean {aspect_ratios.mean():.2f})")
print()
print("  Annotation statistics:")
print(f"    Median persons/img : {np.median(persons_per_img):.0f}")
print(f"    Total person boxes : {len(box_areas):,}")
print(f"    Median box size    : {np.median(box_widths):.0f}×{np.median(box_heights):.0f} px")
print()
print("  Model implications:")
print("    • YOLO  : imgsz=640 comfortably handles median image size.")
print("             Tiny boxes (<5% normalised) may be missed — mosaic aug helps.")
print("    • FRCNN : Most GT boxes fall in anchor scales [64–256]; good coverage.")
print("             Extreme aspect-ratio persons (crowd scenes) may be harder.")
print(f"    • HOG   : {fits_window.mean()*100:.0f}% of boxes ≥ HOG window — good.")
print("             Upsampling needed for remainder; sliding-window at scale=1.5")
print("             should capture medium/large persons well.")
print()
print("✅ Full EDA complete — all plots saved to:", VIZ_DIR.resolve())


---
## Section 3: Model 1 — YOLOv8 Pretrained Baseline

**YOLOv8** (You Only Look Once, version 8) is a state-of-the-art single-stage
object detector from Ultralytics.  It uses a CSPDarknet backbone with a
path aggregation network (PAN) neck, and a decoupled detection head.

In this section we use the **pretrained** YOLOv8n (nano) weights trained on the
full COCO 80-class dataset.  This is our baseline: we do *zero fine-tuning* and
simply run inference on our test set.  This tells us how well the pretrained
model generalises to our specific task out-of-the-box.

**Why YOLOv8n?**  The nano variant runs comfortably on Colab's free T4 GPU and
is fast enough for batch inference.  All findings transfer to larger variants.


In [ ]:
# ─── 3.1  Load pretrained YOLOv8n ─────────────────────────────────────────────
# 'yolov8n.pt' is automatically downloaded from the Ultralytics model hub
# if not already cached.  It contains weights from training on COCO 80 classes.

model_yolo_base = YOLO("yolov8n.pt")
print("✅ YOLOv8n pretrained weights loaded.")
print(f"   Parameter count : {sum(p.numel() for p in model_yolo_base.model.parameters()):,}")


In [ ]:
# ─── 3.2  Run baseline inference on test set ─────────────────────────────────
# We run the pretrained model on every test image and collect predictions.
# We filter to class 0 = "person" only, since the COCO pretrained model
# outputs all 80 classes.
#
# COCO class index 0 in Ultralytics = "person"

def run_yolo_inference(model, img_ids, coco_api, img_dir,
                       conf=CONF_THRESHOLD, person_cls=0):
    """
    Run YOLO inference on a list of COCO image IDs.

    Returns:
        predictions : list of dicts, one per image:
                      {'image_id', 'boxes' [N,4 xyxy], 'scores' [N], 'labels' [N]}
        total_time  : wall-clock seconds for all inference
    """
    predictions = []
    total_time  = 0.0

    for img_id in tqdm(img_ids, desc="  Inference", leave=False):
        img_info = coco_api.loadImgs(img_id)[0]
        img_path = str(img_dir / img_info["file_name"])

        if not Path(img_path).exists():
            continue

        t0 = time.time()
        results = model.predict(img_path, conf=conf, verbose=False, device=DEVICE)
        total_time += time.time() - t0

        res = results[0]
        boxes_all  = res.boxes
        # Keep only person class
        keep = (boxes_all.cls.cpu().numpy().astype(int) == person_cls)
        boxes  = boxes_all.xyxy.cpu().numpy()[keep]   # [N, 4]
        scores = boxes_all.conf.cpu().numpy()[keep]   # [N]
        labels = boxes_all.cls.cpu().numpy()[keep].astype(int)  # [N]

        predictions.append({
            "image_id": img_id,
            "boxes":    boxes,
            "scores":   scores,
            "labels":   labels,
        })

    return predictions, total_time


print("=== Model 1: YOLOv8 Baseline Inference ===")
yolo_base_preds, yolo_base_time = run_yolo_inference(
    model_yolo_base, test_ids, coco_val, VAL_IMG_DIR)

avg_time_base = yolo_base_time / len(yolo_base_preds) * 1000
print(f"✅ Baseline inference complete.")
print(f"   Images processed : {len(yolo_base_preds):,}")
print(f"   Total time       : {yolo_base_time:.1f}s")
print(f"   Avg per image    : {avg_time_base:.1f}ms")


In [ ]:
# ─── 3.3  Evaluate baseline with COCO metrics ────────────────────────────────
# We use pycocotools' COCOeval which implements the official COCO mAP metric.
# mAP@0.5 = mean Average Precision at IoU threshold 0.5
#
# We also compute per-threshold precision & recall for later comparison plots.

def predictions_to_coco_format(predictions, cat_id=1):
    """
    Convert our internal prediction format to the list-of-dicts format
    expected by pycocotools COCOeval.
    """
    coco_results = []
    for pred in predictions:
        img_id = pred["image_id"]
        for bbox_xyxy, score in zip(pred["boxes"], pred["scores"]):
            x1, y1, x2, y2 = bbox_xyxy
            w = float(x2 - x1)
            h = float(y2 - y1)
            coco_results.append({
                "image_id":    int(img_id),
                "category_id": cat_id,
                "bbox":        [float(x1), float(y1), w, h],
                "score":       float(score),
            })
    return coco_results


def run_coco_eval(coco_gt, predictions, img_ids):
    """Run COCOeval and return the stats dict."""
    coco_results = predictions_to_coco_format(predictions)

    if not coco_results:
        print("   ⚠ No predictions to evaluate.")
        return {}

    coco_dt = coco_gt.loadRes(coco_results)
    evaluator = COCOeval(coco_gt, coco_dt, iouType="bbox")
    evaluator.params.imgIds  = img_ids
    evaluator.params.catIds  = [PERSON_CAT_ID]
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    # COCOeval.stats indices:
    # 0: mAP      @[0.50:0.95]
    # 1: mAP@0.50
    # 2: mAP@0.75
    stats = {
        "mAP_50_95": evaluator.stats[0],
        "mAP_50":    evaluator.stats[1],
        "mAP_75":    evaluator.stats[2],
        "AR_1":      evaluator.stats[6],
        "AR_10":     evaluator.stats[7],
        "AR_100":    evaluator.stats[8],
    }
    return stats


print("=== Evaluating Model 1: YOLOv8 Baseline ===")
yolo_base_stats = run_coco_eval(coco_val, yolo_base_preds, test_ids)


In [ ]:
# ─── 3.4  Visualise baseline predictions ─────────────────────────────────────

def visualise_predictions(img_ids, coco_api, img_dir, predictions,
                          n_show=6, title="Model Predictions",
                          save_path=None):
    """Plot images with both ground-truth (green) and predicted (red) boxes."""
    pred_map = {p["image_id"]: p for p in predictions}
    valid_ids = [i for i in img_ids if i in pred_map][:n_show]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(title + "\n🟩 Ground Truth   🟥 Prediction", fontsize=12, fontweight="bold")

    for ax, img_id in zip(axes.flat, valid_ids):
        img_info = coco_api.loadImgs(img_id)[0]
        img_path = img_dir / img_info["file_name"]
        if not img_path.exists():
            ax.axis("off"); continue

        img = np.array(Image.open(img_path).convert("RGB"))
        ax.imshow(img)

        # Ground-truth boxes
        ann_ids = coco_api.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        for ann in coco_api.loadAnns(ann_ids):
            x, y, w, h = ann["bbox"]
            ax.add_patch(mpatches.Rectangle(
                (x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none"))

        # Predicted boxes
        pred = pred_map[img_id]
        for (x1, y1, x2, y2), score in zip(pred["boxes"], pred["scores"]):
            ax.add_patch(mpatches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor="red", facecolor="none"))
            ax.text(x1, y1 - 3, f"{score:.2f}",
                    color="red", fontsize=6, fontweight="bold",
                    bbox=dict(facecolor="white", alpha=0.4, pad=0))

        gt_count   = len(coco_api.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID]))
        pred_count = len(pred["boxes"])
        ax.set_title(f"GT={gt_count} | Pred={pred_count}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()


visualise_predictions(
    test_ids, coco_val, VAL_IMG_DIR, yolo_base_preds,
    title="Model 1 — YOLOv8 Pretrained Baseline",
    save_path=VIZ_DIR / "01_yolo_baseline_predictions.png")


---
## Section 3.5 / 4.5 / 5.5 / 6.6 — Explainability & Calibration

Each model section includes:
- **Explainability:** What does the model "look at"?  
  GradCAM-style saliency (YOLO/FRCNN) or SVM weight maps (HOG+SVM).
- **Calibration:** Are confidence scores well-calibrated?  
  A reliability diagram plots *predicted confidence* vs *empirical accuracy*.
  Expected Calibration Error (ECE) quantifies the gap.

> *A well-calibrated model that predicts 0.8 confidence should be correct ~80% of the time.*


In [ ]:
# ─── Shared Calibration Utility ──────────────────────────────────────────────
# This cell defines helper functions used by ALL model calibration sections.
# Run once after Section 2.

def compute_detection_calibration(predictions, gt_coco, img_ids,
                                  iou_thresh=IOU_THRESHOLD, n_bins=10):
    """
    Reliability diagram data for detection models.

    For each predicted box we record (confidence, is_tp) where is_tp=1 if the
    box matches a GT box at IoU >= iou_thresh.  We then bin by confidence and
    compute the empirical TP rate per bin.

    Returns:
        bin_centres : np.ndarray [n_bins]
        bin_acc     : np.ndarray [n_bins]  — empirical TP rate per bin
        bin_counts  : np.ndarray [n_bins]  — # predictions per bin
        ece         : float                — Expected Calibration Error
    """
    all_conf, all_tp = [], []
    pred_map = {p["image_id"]: p for p in predictions}

    for img_id in img_ids:
        ann_ids = gt_coco.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        anns    = [a for a in gt_coco.loadAnns(ann_ids) if a["area"] >= MIN_AREA]
        gt_xyxy = np.array([[a["bbox"][0], a["bbox"][1],
                              a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]]
                             for a in anns], dtype=float) if anns else np.zeros((0,4))

        pred = pred_map.get(img_id)
        if pred is None or len(pred["boxes"]) == 0:
            continue

        matched_gt = set()
        order = np.argsort(-pred["scores"])
        for idx in order:
            box   = pred["boxes"][idx]
            score = pred["scores"][idx]
            all_conf.append(float(score))

            if gt_xyxy.shape[0] == 0:
                all_tp.append(0); continue

            x1 = np.maximum(box[0], gt_xyxy[:,0])
            y1 = np.maximum(box[1], gt_xyxy[:,1])
            x2 = np.minimum(box[2], gt_xyxy[:,2])
            y2 = np.minimum(box[3], gt_xyxy[:,3])
            inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
            ap = (box[2]-box[0])*(box[3]-box[1])
            ag = (gt_xyxy[:,2]-gt_xyxy[:,0])*(gt_xyxy[:,3]-gt_xyxy[:,1])
            ious = inter / np.maximum(ap + ag - inter, 1e-6)
            best = np.argmax(ious)
            if ious[best] >= iou_thresh and best not in matched_gt:
                matched_gt.add(best); all_tp.append(1)
            else:
                all_tp.append(0)

    all_conf = np.array(all_conf)
    all_tp   = np.array(all_tp)

    bins = np.linspace(0, 1, n_bins + 1)
    bin_centres = (bins[:-1] + bins[1:]) / 2
    bin_acc, bin_counts = np.zeros(n_bins), np.zeros(n_bins)

    for i in range(n_bins):
        mask = (all_conf >= bins[i]) & (all_conf < bins[i+1])
        if mask.sum() > 0:
            bin_acc[i]    = all_tp[mask].mean()
            bin_counts[i] = mask.sum()

    # ECE: weighted average of |confidence - accuracy|
    total = bin_counts.sum()
    ece   = (bin_counts / max(total, 1) * np.abs(bin_centres - bin_acc)).sum()

    return bin_centres, bin_acc, bin_counts, ece


def plot_reliability_diagram(bin_centres, bin_acc, bin_counts, ece,
                             title="Reliability Diagram", save_path=None, ax=None):
    """Plot a reliability diagram with a gap-fill between ideal and actual."""
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(6, 5))

    # Perfect calibration line
    ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")

    # Actual calibration bars
    bar_w = bin_centres[1] - bin_centres[0] if len(bin_centres) > 1 else 0.1
    ax.bar(bin_centres, bin_acc,
           width=bar_w * 0.8, alpha=0.7, color="#4C72B0",
           edgecolor="white", label="Model accuracy")

    # Gap fill (over/under confidence)
    for i, (bc, ba) in enumerate(zip(bin_centres, bin_acc)):
        if ba < bc:
            ax.bar(bc, bc - ba, bottom=ba, width=bar_w*0.8,
                   alpha=0.3, color="#C44E52")
        else:
            ax.bar(bc, ba - bc, bottom=bc, width=bar_w*0.8,
                   alpha=0.3, color="#55A868")

    ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
    ax.set_xlabel("Predicted Confidence"); ax.set_ylabel("Empirical TP Rate")
    ax.set_title(f"{title}\nECE = {ece:.4f}", fontsize=10, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    if standalone:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.show()


print("✅ Calibration utility functions defined.")


### 3.5 — Model 1 Explainability & Calibration (YOLOv8 Pretrained Baseline)


In [ ]:
# ─── 3.5.1  YOLOv8 Baseline — GradCAM Saliency ───────────────────────────────
# Ultralytics provides a built-in Grad-CAM implementation via the
# `ultralytics.utils.plotting` module.  We compute feature-map activations
# for the detection layers to visualise what the model attends to.
#
# We use a manual approach with hooks on the last backbone feature map
# to keep dependencies minimal (no external GradCAM libraries).

import torch.nn.functional as F

def compute_yolo_gradcam(model, img_path, person_cls=0, conf=0.25):
    """
    Compute a simple activation map from the last backbone feature layer.
    This is an *activation* map (not gradient-weighted) — fast and reliable.
    Returns: (original_image_np, saliency_map_np)
    """
    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    # Store activations from the last backbone layer
    activations = {}
    def hook_fn(module, inp, out):
        activations["feat"] = out.detach()

    # Register hook on the last Conv block of the YOLOv8 backbone
    # (the model.model[-2] is typically the SPPF layer)
    try:
        target_layer = model.model.model[-2]
    except (AttributeError, IndexError):
        target_layer = list(model.model.parameters())[-1]

    handle = target_layer.register_forward_hook(hook_fn) if hasattr(target_layer, "register_forward_hook") else None

    with torch.no_grad():
        result = model.predict(str(img_path), conf=conf, verbose=False, device=DEVICE)

    if handle:
        handle.remove()

    # Build saliency from stored activations
    if "feat" in activations:
        feat = activations["feat"]
        if feat.dim() == 4:
            # Average across channels → spatial heatmap
            saliency = feat[0].mean(dim=0).cpu().numpy()
            saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
            # Upsample to image size
            import cv2 as _cv2_optional
            saliency_resized = np.array(
                Image.fromarray((saliency * 255).astype(np.uint8)).resize(
                    (img_np.shape[1], img_np.shape[0]), Image.BILINEAR)) / 255.0
        else:
            saliency_resized = np.ones((img_np.shape[0], img_np.shape[1]))
    else:
        saliency_resized = np.ones((img_np.shape[0], img_np.shape[1]))

    return img_np, saliency_resized, result[0]


print("=== Model 1: YOLOv8 Baseline — Explainability ===")

sample_explain = random.sample(test_ids[:MAX_TEST_IMAGES], min(4, len(test_ids)))
fig, axes = plt.subplots(len(sample_explain), 3,
                          figsize=(15, 4 * len(sample_explain)))
if len(sample_explain) == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("Model 1 — YOLOv8 Baseline: Activation Saliency Map\n"
             "Left: Original | Centre: Saliency Overlay | Right: Detections",
             fontsize=12, fontweight="bold")

for row, img_id in enumerate(sample_explain):
    info     = coco_val.loadImgs(img_id)[0]
    img_path = VAL_IMG_DIR / info["file_name"]
    if not img_path.exists():
        continue

    img_np, saliency, result = compute_yolo_gradcam(model_yolo_base, img_path)

    axes[row, 0].imshow(img_np)
    axes[row, 0].axis("off")
    axes[row, 0].set_title("Original", fontsize=9)

    # Saliency overlay
    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(saliency, cmap="jet", alpha=0.45)
    axes[row, 1].axis("off")
    axes[row, 1].set_title("Activation Map (jet overlay)", fontsize=9)

    # Detections
    det_img = img_np.copy()
    axes[row, 2].imshow(det_img)
    for b in result.boxes:
        if int(b.cls) == 0:
            x1,y1,x2,y2 = b.xyxy[0].cpu().numpy()
            axes[row, 2].add_patch(mpatches.Rectangle(
                (x1,y1), x2-x1, y2-y1, lw=2, edgecolor="red", fc="none"))
            axes[row, 2].text(x1, y1-3, f"{b.conf[0]:.2f}",
                              color="red", fontsize=7, fontweight="bold")
    axes[row, 2].axis("off")
    axes[row, 2].set_title("Person Detections", fontsize=9)

plt.tight_layout()
plt.savefig(VIZ_DIR / "EXPLAIN_01_yolo_baseline_saliency.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ YOLOv8 Baseline explainability complete.")


In [ ]:
# ─── 3.5.2  YOLOv8 Baseline — Calibration ────────────────────────────────────
# A well-calibrated detector's confidence score should match its empirical
# True Positive rate.  We use our custom reliability diagram tool.

print("=== Model 1: YOLOv8 Baseline — Calibration ===")

bc_base, ba_base, bcount_base, ece_base = compute_detection_calibration(
    yolo_base_preds, coco_val, test_ids[:MAX_TEST_IMAGES], n_bins=10)

print(f"  ECE (Expected Calibration Error): {ece_base:.4f}")
print(f"  (0 = perfect calibration, 1 = completely wrong)")

# Confidence score distribution
all_scores_base = np.concatenate([p["scores"] for p in yolo_base_preds if len(p["scores"])])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model 1 — YOLOv8 Pretrained Baseline: Calibration",
             fontsize=12, fontweight="bold")

plot_reliability_diagram(bc_base, ba_base, bcount_base, ece_base,
                         title="Reliability Diagram", ax=axes[0])

axes[1].hist(all_scores_base, bins=40, color="#4C72B0", edgecolor="white", lw=0.5)
axes[1].set_title("Confidence Score Distribution")
axes[1].set_xlabel("Confidence"); axes[1].set_ylabel("# predictions")
axes[1].axvline(CONF_THRESHOLD, color="red", ls="--",
                label=f"Threshold ({CONF_THRESHOLD})")
axes[1].legend()

plt.tight_layout()
plt.savefig(VIZ_DIR / "CALIB_01_yolo_baseline.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ Calibration complete.  ECE = {ece_base:.4f}")


---
## Section 4: Model 2 — YOLOv8 Fine-Tuned

Fine-tuning adapts a pretrained model to a specific task.  Starting from the
COCO-pretrained weights (which already know what people look like), we continue
training on our filtered person-only dataset.

**Training strategy (two-phase):**
1. **Phase 1 — Backbone frozen (5 epochs):** Only the detection head trains.
   This lets the new head settle before we risk destroying the rich backbone
   features with large gradients.
2. **Phase 2 — Full fine-tune (10 epochs):** All layers train with a very low
   learning rate so we refine without catastrophic forgetting.

**Augmentations:** We use Ultralytics' built-in augmentation pipeline which
includes horizontal flip, scale jitter, HSV colour augmentation, and mosaic
(combining 4 images) — all designed specifically for detection tasks.


In [ ]:
# ─── 4.1  Phase 1: Train detection head only (backbone frozen) ───────────────
# Before training, we check for an existing Phase 1 checkpoint.
# If found, we skip training and load the saved weights directly.
# This prevents redundant training when the notebook is re-run.

YOLO_FINETUNE_DIR = (MODEL_DIR / "yolo_finetune").resolve()
YOLO_FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

phase1_best = YOLO_FINETUNE_DIR / "phase1" / "weights" / "best.pt"

if phase1_best.exists():
    print(f"✅ Phase 1 checkpoint found — skipping training.")
    print(f"   Loading: {phase1_best}")
    model_yolo_ft = YOLO(str(phase1_best))
else:
    print("=== Phase 1: Training detection head (backbone frozen) ===")
    model_yolo_ft = YOLO("yolov8n.pt")   # start from COCO pretrained weights

    results_phase1 = model_yolo_ft.train(
        data    = str(yolo_yaml_path),
        epochs  = 5,
        imgsz   = 640,
        batch   = 16,
        lr0     = 0.001,
        lrf     = 0.01,
        freeze  = 10,          # freeze first 10 backbone layers
        mosaic  = 1.0,
        flipud  = 0.0,
        fliplr  = 0.5,
        scale   = 0.5,
        hsv_h   = 0.015,
        hsv_s   = 0.7,
        hsv_v   = 0.4,
        project = str(YOLO_FINETUNE_DIR),
        name    = "phase1",
        exist_ok= True,
        verbose = False,
        device  = DEVICE,
        seed    = SEED,
        workers = 2,
    )
    print(f"✅ Phase 1 complete. Save dir: {results_phase1.save_dir}")

print(f"   Phase 1 weights: {phase1_best}")


In [ ]:
# ─── 4.2  Phase 2: Full fine-tune (all layers, lower LR) ────────────────────
# If a Phase 2 checkpoint already exists, load it directly.
# Otherwise we continue from the Phase 1 checkpoint (resume training).

phase2_run  = YOLO_FINETUNE_DIR / "phase2"
yolo_ft_best = phase2_run / "weights" / "best.pt"

if yolo_ft_best.exists():
    print(f"✅ Phase 2 checkpoint found — skipping training.")
    print(f"   Loading: {yolo_ft_best}")
    model_yolo_ft = YOLO(str(yolo_ft_best))
    phase2_run_dir = phase2_run
else:
    print("=== Phase 2: Full fine-tune (all layers) ===")
    # Load the Phase 1 best checkpoint
    model_yolo_ft = YOLO(str(phase1_best))

    results_phase2 = model_yolo_ft.train(
        data    = str(yolo_yaml_path),
        epochs  = 10,
        imgsz   = 640,
        batch   = 16,
        lr0     = 0.0001,      # lower LR to preserve backbone features
        lrf     = 0.01,
        freeze  = 0,           # unfreeze all layers
        mosaic  = 1.0,
        fliplr  = 0.5,
        scale   = 0.5,
        hsv_h   = 0.015,
        hsv_s   = 0.7,
        hsv_v   = 0.4,
        project = str(YOLO_FINETUNE_DIR),
        name    = "phase2",
        exist_ok= True,
        verbose = False,
        device  = DEVICE,
        seed    = SEED,
        workers = 2,
    )
    phase2_run_dir = Path(results_phase2.save_dir)
    yolo_ft_best   = phase2_run_dir / "weights" / "best.pt"
    print(f"✅ Phase 2 complete. Best weights: {yolo_ft_best}")

print(f"   Fine-tuned model: {yolo_ft_best}")


In [ ]:
# ─── 4.3  Evaluate fine-tuned model on test set ──────────────────────────────

model_yolo_ft_eval = YOLO(str(yolo_ft_best))
print("=== Model 2: YOLOv8 Fine-Tuned Inference ===")

yolo_ft_preds, yolo_ft_time = run_yolo_inference(
    model_yolo_ft_eval, test_ids, coco_val, VAL_IMG_DIR)

avg_time_ft = yolo_ft_time / len(yolo_ft_preds) * 1000
print(f"✅ Fine-tuned inference complete.")
print(f"   Avg per image : {avg_time_ft:.1f}ms")

print("\n=== Evaluating Model 2: YOLOv8 Fine-Tuned ===")
yolo_ft_stats = run_coco_eval(coco_val, yolo_ft_preds, test_ids)


In [ ]:
# ─── 4.4  Visualise training curves ──────────────────────────────────────────
# Ultralytics saves a CSV with per-epoch metrics during training.

def plot_yolo_training_curves(run_dir, title="Training Curves"):
    """Load and plot Ultralytics training metrics CSV."""
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        print(f"   ⚠ CSV not found at {csv_path}")
        return

    import csv
    metrics = defaultdict(list)
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            for k, v in row.items():
                k = k.strip()
                try:
                    metrics[k].append(float(v))
                except (ValueError, TypeError):
                    pass

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title, fontsize=12, fontweight="bold")

    # Plot box loss
    if "train/box_loss" in metrics:
        axes[0].plot(metrics["train/box_loss"], label="Train box loss", color="steelblue")
        if "val/box_loss" in metrics:
            axes[0].plot(metrics["val/box_loss"],  label="Val box loss",   color="orange")
        axes[0].set_title("Box Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

    # Plot class loss
    if "train/cls_loss" in metrics:
        axes[1].plot(metrics["train/cls_loss"], label="Train cls loss", color="steelblue")
        if "val/cls_loss" in metrics:
            axes[1].plot(metrics["val/cls_loss"],  label="Val cls loss",   color="orange")
        axes[1].set_title("Classification Loss"); axes[1].legend(); axes[1].set_xlabel("Epoch")

    # Plot mAP
    map_key = "metrics/mAP50(B)"
    if map_key in metrics:
        axes[2].plot(metrics[map_key], label="mAP@0.5", color="green")
        axes[2].set_title("mAP@0.5"); axes[2].legend(); axes[2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.savefig(VIZ_DIR / "02_yolo_finetune_curves.png", dpi=120, bbox_inches="tight")
    plt.show()

plot_yolo_training_curves(
    phase2_run_dir,
    title="Model 2 — YOLOv8 Fine-Tune Training Curves (Phase 2)")

# Visualise predictions
visualise_predictions(
    test_ids, coco_val, VAL_IMG_DIR, yolo_ft_preds,
    title="Model 2 — YOLOv8 Fine-Tuned",
    save_path=VIZ_DIR / "03_yolo_finetuned_predictions.png")

### 4.5 — Model 2 Explainability & Calibration (YOLOv8 Fine-Tuned)


In [ ]:
# ─── 4.5.1  YOLOv8 Fine-Tuned — Saliency Maps ───────────────────────────────
# We compare activation maps from Baseline vs Fine-Tuned on the same images
# to visualise how fine-tuning shifted the model's attention.

print("=== Model 2: YOLOv8 Fine-Tuned — Explainability ===")

model_yolo_ft_eval = YOLO(str(yolo_ft_best))

fig, axes = plt.subplots(len(sample_explain), 3,
                          figsize=(15, 4 * len(sample_explain)))
if len(sample_explain) == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("Model 2 — YOLOv8 Fine-Tuned: Activation Saliency Map\n"
             "Left: Original | Centre: Saliency Overlay | Right: Detections",
             fontsize=12, fontweight="bold")

for row, img_id in enumerate(sample_explain):
    info     = coco_val.loadImgs(img_id)[0]
    img_path = VAL_IMG_DIR / info["file_name"]
    if not img_path.exists():
        continue

    img_np, saliency, result = compute_yolo_gradcam(model_yolo_ft_eval, img_path)

    axes[row, 0].imshow(img_np); axes[row, 0].axis("off")
    axes[row, 0].set_title("Original", fontsize=9)

    axes[row, 1].imshow(img_np)
    axes[row, 1].imshow(saliency, cmap="jet", alpha=0.45)
    axes[row, 1].axis("off")
    axes[row, 1].set_title("Activation Map (fine-tuned)", fontsize=9)

    axes[row, 2].imshow(img_np)
    for b in result.boxes:
        if int(b.cls) == 0:
            x1,y1,x2,y2 = b.xyxy[0].cpu().numpy()
            axes[row, 2].add_patch(mpatches.Rectangle(
                (x1,y1),x2-x1,y2-y1,lw=2,edgecolor="red",fc="none"))
            axes[row, 2].text(x1,y1-3,f"{b.conf[0]:.2f}",
                              color="red",fontsize=7,fontweight="bold")
    axes[row, 2].axis("off")
    axes[row, 2].set_title("Person Detections", fontsize=9)

plt.tight_layout()
plt.savefig(VIZ_DIR / "EXPLAIN_02_yolo_ft_saliency.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ YOLOv8 Fine-Tuned explainability complete.")


In [ ]:
# ─── 4.5.2  YOLOv8 Fine-Tuned — Calibration ──────────────────────────────────

print("=== Model 2: YOLOv8 Fine-Tuned — Calibration ===")

bc_ft, ba_ft, bcount_ft, ece_ft = compute_detection_calibration(
    yolo_ft_preds, coco_val, test_ids[:MAX_TEST_IMAGES], n_bins=10)

all_scores_ft = np.concatenate([p["scores"] for p in yolo_ft_preds if len(p["scores"])])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model 2 — YOLOv8 Fine-Tuned: Calibration", fontsize=12, fontweight="bold")

plot_reliability_diagram(bc_ft, ba_ft, bcount_ft, ece_ft,
                         title="Reliability Diagram", ax=axes[0])

axes[1].hist(all_scores_ft, bins=40, color="#FF9800", edgecolor="white", lw=0.5)
axes[1].set_title("Confidence Score Distribution (Fine-Tuned)")
axes[1].set_xlabel("Confidence"); axes[1].set_ylabel("# predictions")
axes[1].axvline(CONF_THRESHOLD, color="red", ls="--", label=f"Threshold ({CONF_THRESHOLD})")
axes[1].legend()

plt.tight_layout()
plt.savefig(VIZ_DIR / "CALIB_02_yolo_ft.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"  ECE (Fine-Tuned): {ece_ft:.4f}  |  ECE (Baseline): {ece_base:.4f}")
print(f"  Calibration {'improved' if ece_ft < ece_base else 'worsened'} after fine-tuning "
      f"(Δ ECE = {ece_ft - ece_base:+.4f})")


---
## Section 5: Model 3 — Faster R-CNN (PyTorch)

Faster R-CNN is a two-stage detector:
1. **Stage 1 — Region Proposal Network (RPN):** A small FCN that scans the
   feature map and proposes candidate object regions (anchors).
2. **Stage 2 — Detection Head:** Each proposal is classified and its box
   is refined.

We load the **pretrained ResNet-50 + FPN** backbone (trained on COCO) from
torchvision, then replace the classification head with a 2-class head
(background + person).  We fine-tune for a small number of epochs, which is
sufficient because the backbone features are already very rich.

**Why Faster R-CNN alongside YOLO?**  Faster R-CNN is generally more accurate
(especially for small objects) but slower.  This comparison lets us analyse the
accuracy–speed trade-off empirically.


In [ ]:
# ─── 5.1  Build Faster R-CNN model with custom head ─────────────────────────

def build_faster_rcnn(num_classes: int = 2) -> nn.Module:
    """
    Load pretrained Faster R-CNN ResNet-50 FPN and replace its head.

    num_classes includes the background class, so:
        num_classes = 2  →  background (0) + person (1)
    """
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1
    model   = fasterrcnn_resnet50_fpn(weights=weights)

    # Replace the box predictor (classification + regression head)
    # in_features is the size of the ROI-pooled feature vector
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model


frcnn_model = build_faster_rcnn(num_classes=2)
frcnn_model = frcnn_model.to(DEVICE)

total_params = sum(p.numel() for p in frcnn_model.parameters())
trainable    = sum(p.numel() for p in frcnn_model.parameters() if p.requires_grad)
print(f"✅ Faster R-CNN built.")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable:,}")


In [ ]:
# ─── 5.2  Data loaders for Faster R-CNN ──────────────────────────────────────
# We use a small batch size because Faster R-CNN is memory-intensive —
# each image can have a different size and the model stores intermediate
# feature maps for every proposed region.

FRCNN_BATCH = 4   # adjust down to 2 if you get OOM errors
# Use num_workers=0 in notebook/Windows to avoid worker spawn/hangs
frcnn_train_loader = DataLoader(
    train_dataset,
    batch_size  = FRCNN_BATCH,
    shuffle     = True,
    num_workers = 0,
    pin_memory  = (DEVICE.type == "cuda"),
    collate_fn  = collate_fn,   # required because images have different sizes
)

frcnn_val_loader = DataLoader(
    val_dataset,
    batch_size  = 1,            # batch=1 at val time avoids padding issues
    shuffle     = False,
    num_workers = 0,
    pin_memory  = (DEVICE.type == "cuda"),
    collate_fn  = collate_fn,
)

print(f"✅ Faster R-CNN data loaders ready.")
print(f"   Train batches : {len(frcnn_train_loader):,}  (batch size {FRCNN_BATCH})")
print(f"   Val batches   : {len(frcnn_val_loader):,}  (batch size 1)")

# --- Quick single-batch diagnostic (run to measure one forward+loss step) ---
print('--- Single-batch diagnostic (num_workers=0) ---')
try:
    it = iter(frcnn_train_loader)
    images, targets = next(it)
except StopIteration:
    print('⚠ Train loader is empty.')
else:
    # Move inputs to device and report if first batch has no boxes
    targets_device = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
    images_device = [img.to(DEVICE) for img in images]
    if any(t['boxes'].shape[0] == 0 for t in targets_device):
        print('⚠ First batch contains images with zero boxes after filtering.')
    else:
        frcnn_model.train()
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        loss_dict = frcnn_model(images_device, targets_device)
        total_loss = sum(loss for loss in loss_dict.values())
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        print('One forward+loss time (s):', time.time() - t0)
        try:
            print({k: float(v.detach().cpu().item()) for k, v in loss_dict.items()})
        except Exception:
            pass


In [ ]:
# ─── 5.3  Training loop ───────────────────────────────────────────────────────
# If a saved checkpoint already exists, we load it and skip training.
# If a checkpoint exists but we want to continue training, we resume from it.

FRCNN_EPOCHS = 5
FRCNN_LR     = 5e-4
best_frcnn_path = MODEL_DIR / "frcnn_best.pth"
frcnn_ckpt_path = MODEL_DIR / "frcnn_checkpoint.pth"   # latest checkpoint for resuming

optimizer = optim.SGD(
    [p for p in frcnn_model.parameters() if p.requires_grad],
    lr=FRCNN_LR, momentum=0.9, weight_decay=1e-4)
lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

frcnn_train_losses = []
frcnn_val_losses   = []
start_epoch        = 1
best_val_loss      = float("inf")


def train_one_epoch(model, loader, optimiser, device):
    """One training epoch; returns mean total loss."""
    model.train()
    epoch_loss = 0.0; n_batches = 0
    total_batches = max(len(loader), 1)
    start_time = time.time()
    progress = tqdm(total=total_batches, desc="    Train", leave=True, unit="batch", dynamic_ncols=True)
    for batch_idx, (images, targets) in enumerate(loader, start=1):
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        if any(t["boxes"].shape[0] == 0 for t in targets):
            progress.update(1); continue
        loss_dict  = model(images, targets)
        total_loss = sum(loss for loss in loss_dict.values())
        optimiser.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimiser.step()
        epoch_loss += total_loss.item(); n_batches += 1
        avg_loss = epoch_loss / max(n_batches, 1)
        progress.update(1)
        progress.set_postfix({"batch": f"{batch_idx}/{total_batches}",
                              "loss": f"{avg_loss:.4f}",
                              "elapsed": f"{time.time()-start_time:.0f}s"})
    progress.close()
    return epoch_loss / max(n_batches, 1)


def validate_one_epoch(model, loader, device):
    """Validation pass in training mode to compute loss."""
    model.train()
    epoch_loss = 0.0; n_batches = 0
    total_batches = max(len(loader), 1)
    progress = tqdm(total=total_batches, desc="    Val  ", leave=True, unit="batch", dynamic_ncols=True)
    with torch.no_grad():
        for batch_idx, (images, targets) in enumerate(loader, start=1):
            images  = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            if any(t["boxes"].shape[0] == 0 for t in targets):
                progress.update(1); continue
            loss_dict  = model(images, targets)
            total_loss = sum(loss for loss in loss_dict.values())
            epoch_loss += total_loss.item(); n_batches += 1
            progress.update(1)
            progress.set_postfix({"loss": f"{epoch_loss/max(n_batches,1):.4f}"})
    progress.close()
    return epoch_loss / max(n_batches, 1)


# ── Load checkpoint if available ──────────────────────────────────────────────
if best_frcnn_path.exists() and not frcnn_ckpt_path.exists():
    # Best weights exist but no in-progress checkpoint → training is complete
    print(f"✅ Faster R-CNN weights found — skipping training.")
    print(f"   Loading: {best_frcnn_path}")
    frcnn_model.load_state_dict(torch.load(best_frcnn_path, map_location=DEVICE))
elif frcnn_ckpt_path.exists():
    # Resume from latest checkpoint
    ckpt = torch.load(frcnn_ckpt_path, map_location=DEVICE)
    frcnn_model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    lr_scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch     = ckpt["epoch"] + 1
    best_val_loss   = ckpt.get("best_val_loss", float("inf"))
    frcnn_train_losses = ckpt.get("train_losses", [])
    frcnn_val_losses   = ckpt.get("val_losses",   [])
    print(f"✅ Resuming Faster R-CNN training from epoch {start_epoch}.")
else:
    print(f"=== Training Faster R-CNN for {FRCNN_EPOCHS} epochs ===")

if start_epoch <= FRCNN_EPOCHS and not (best_frcnn_path.exists() and not frcnn_ckpt_path.exists()):
    for epoch in range(start_epoch, FRCNN_EPOCHS + 1):
        t0         = time.time()
        train_loss = train_one_epoch(frcnn_model, frcnn_train_loader, optimizer, DEVICE)
        val_loss   = validate_one_epoch(frcnn_model, frcnn_val_loader, DEVICE)
        lr_scheduler.step()
        frcnn_train_losses.append(train_loss)
        frcnn_val_losses.append(val_loss)
        elapsed = time.time() - t0
        print(f"  Epoch {epoch}/{FRCNN_EPOCHS} | Train loss: {train_loss:.4f} | "
              f"Val loss: {val_loss:.4f} | LR: {lr_scheduler.get_last_lr()[0]:.2e} | "
              f"Time: {elapsed:.0f}s")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(frcnn_model.state_dict(), best_frcnn_path)
            print(f"  💾 Best model saved (val loss {best_val_loss:.4f})")
        # Save rolling checkpoint for potential resume
        torch.save({
            "epoch":           epoch,
            "model_state":     frcnn_model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": lr_scheduler.state_dict(),
            "best_val_loss":   best_val_loss,
            "train_losses":    frcnn_train_losses,
            "val_losses":      frcnn_val_losses,
        }, frcnn_ckpt_path)
    # Training complete — remove rolling checkpoint to signal completion
    frcnn_ckpt_path.unlink(missing_ok=True)
    print(f"\n✅ Faster R-CNN training complete.  Best val loss: {best_val_loss:.4f}")


In [ ]:
# ─── 5.4  Faster R-CNN inference on test set ─────────────────────────────────

# Load best checkpoint
frcnn_model.load_state_dict(torch.load(best_frcnn_path, map_location=DEVICE))
frcnn_model.eval()
print("✅ Best Faster R-CNN weights loaded.")

def run_frcnn_inference(model, img_ids, coco_api, img_dir,
                        conf=CONF_THRESHOLD, device=DEVICE):
    """Run Faster R-CNN inference and return predictions in our standard format."""
    model.eval()
    predictions = []
    total_time  = 0.0
    to_tensor   = T.ToTensor()

    for img_id in tqdm(img_ids, desc="  Inference", leave=False):
        img_info = coco_api.loadImgs(img_id)[0]
        img_path = img_dir / img_info["file_name"]
        if not img_path.exists():
            continue

        img_tensor = to_tensor(Image.open(img_path).convert("RGB")).to(device)

        t0 = time.time()
        with torch.no_grad():
            outputs = model([img_tensor])
        total_time += time.time() - t0

        out    = outputs[0]
        keep   = (out["scores"].cpu().numpy() >= conf) &                  (out["labels"].cpu().numpy() == 1)  # class 1 = person

        predictions.append({
            "image_id": img_id,
            "boxes":    out["boxes"].cpu().numpy()[keep],
            "scores":   out["scores"].cpu().numpy()[keep],
            "labels":   out["labels"].cpu().numpy()[keep],
        })

    return predictions, total_time


print("=== Model 3: Faster R-CNN Inference ===")
frcnn_preds, frcnn_time = run_frcnn_inference(
    frcnn_model, test_ids, coco_val, VAL_IMG_DIR)

avg_time_frcnn = frcnn_time / len(frcnn_preds) * 1000
print(f"✅ Done.  Avg per image: {avg_time_frcnn:.1f}ms")

print("\n=== Evaluating Model 3: Faster R-CNN ===")
frcnn_stats = run_coco_eval(coco_val, frcnn_preds, test_ids)

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Model 3 — Faster R-CNN Training Curves", fontsize=12, fontweight="bold")
axes[0].plot(frcnn_train_losses, label="Train loss", color="steelblue", marker="o")
axes[0].plot(frcnn_val_losses,   label="Val loss",   color="orange",    marker="o")
axes[0].set_title("Total Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[1].plot(frcnn_train_losses, color="steelblue", marker="o")
axes[1].set_title("Train Loss (zoomed)"); axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(VIZ_DIR / "04_frcnn_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

visualise_predictions(
    test_ids, coco_val, VAL_IMG_DIR, frcnn_preds,
    title="Model 3 — Faster R-CNN",
    save_path=VIZ_DIR / "05_frcnn_predictions.png")


### 5.5 — Model 3 Explainability & Calibration (Faster R-CNN)


In [ ]:
# ─── 5.5.1  Faster R-CNN — GradCAM on Backbone ───────────────────────────────
# We register a forward hook on the last ResNet layer4 feature map.
# The per-channel mean gives a spatial heatmap of where the backbone
# concentrates its representation — a proxy for "where the model looks".

print("=== Model 3: Faster R-CNN — Explainability ===")

frcnn_model.eval()
frcnn_model.load_state_dict(torch.load(best_frcnn_path, map_location=DEVICE))

def frcnn_activation_map(model, img_path, device=DEVICE):
    """
    Forward hook on ResNet layer4 to get activation saliency.
    Returns: (img_np, saliency_np [0,1], raw_output)
    """
    img    = Image.open(img_path).convert("RGB")
    img_np = np.array(img)
    tensor = T.ToTensor()(img).to(device)

    activations = {}
    def hook(m, i, o):
        activations["feat"] = o.detach()

    # Hook on the last residual block of ResNet-50
    handle = model.backbone.body.layer4.register_forward_hook(hook)

    with torch.no_grad():
        outputs = model([tensor])

    handle.remove()

    feat = activations.get("feat")
    if feat is not None and feat.dim() == 4:
        saliency = feat[0].mean(dim=0).cpu().numpy()
        saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
        saliency = np.array(
            Image.fromarray((saliency*255).astype(np.uint8)).resize(
                (img_np.shape[1], img_np.shape[0]), Image.BILINEAR)) / 255.0
    else:
        saliency = np.ones((img_np.shape[0], img_np.shape[1]))

    return img_np, saliency, outputs[0]


fig, axes = plt.subplots(len(sample_explain), 3,
                          figsize=(15, 4 * len(sample_explain)))
if len(sample_explain) == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("Model 3 — Faster R-CNN: Backbone Activation Map\n"
             "Left: Original | Centre: Saliency Overlay | Right: Detections",
             fontsize=12, fontweight="bold")

for row, img_id in enumerate(sample_explain):
    info     = coco_val.loadImgs(img_id)[0]
    img_path = VAL_IMG_DIR / info["file_name"]
    if not img_path.exists():
        continue

    img_np, saliency, output = frcnn_activation_map(frcnn_model, img_path)
    keep = (output["scores"].cpu().numpy() >= CONF_THRESHOLD) &            (output["labels"].cpu().numpy() == 1)
    pred_boxes  = output["boxes"].cpu().numpy()[keep]
    pred_scores = output["scores"].cpu().numpy()[keep]

    axes[row,0].imshow(img_np); axes[row,0].axis("off")
    axes[row,0].set_title("Original", fontsize=9)

    axes[row,1].imshow(img_np)
    axes[row,1].imshow(saliency, cmap="hot", alpha=0.5)
    axes[row,1].axis("off")
    axes[row,1].set_title("ResNet Layer4 Activation", fontsize=9)

    axes[row,2].imshow(img_np)
    for (x1,y1,x2,y2), sc in zip(pred_boxes, pred_scores):
        axes[row,2].add_patch(mpatches.Rectangle(
            (x1,y1),x2-x1,y2-y1,lw=2,edgecolor="red",fc="none"))
        axes[row,2].text(x1,y1-3,f"{sc:.2f}",
                         color="red",fontsize=7,fontweight="bold")
    axes[row,2].axis("off")
    axes[row,2].set_title("Person Detections", fontsize=9)

plt.tight_layout()
plt.savefig(VIZ_DIR / "EXPLAIN_03_frcnn_activation.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Faster R-CNN explainability complete.")


In [ ]:
# ─── 5.5.2  Faster R-CNN — Calibration ────────────────────────────────────────
# Faster R-CNN uses a softmax on the ROI classifier head.  The resulting
# scores tend to be *overconfident* because Softmax peaks are sharp.

print("=== Model 3: Faster R-CNN — Calibration ===")

bc_frcnn, ba_frcnn, bcount_frcnn, ece_frcnn = compute_detection_calibration(
    frcnn_preds, coco_val, test_ids[:MAX_TEST_IMAGES], n_bins=10)

all_scores_frcnn = np.concatenate([p["scores"] for p in frcnn_preds if len(p["scores"])])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model 3 — Faster R-CNN: Calibration", fontsize=12, fontweight="bold")

plot_reliability_diagram(bc_frcnn, ba_frcnn, bcount_frcnn, ece_frcnn,
                         title="Reliability Diagram", ax=axes[0])

axes[1].hist(all_scores_frcnn, bins=40, color="#4CAF50", edgecolor="white", lw=0.5)
axes[1].set_title("Confidence Score Distribution (Faster R-CNN)")
axes[1].set_xlabel("Confidence"); axes[1].set_ylabel("# predictions")
axes[1].axvline(CONF_THRESHOLD, color="red", ls="--", label=f"Threshold ({CONF_THRESHOLD})")
axes[1].legend()

plt.tight_layout()
plt.savefig(VIZ_DIR / "CALIB_03_frcnn.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"  ECE (Faster R-CNN): {ece_frcnn:.4f}")
print("  Note: Faster R-CNN often over-confident (softmax head). "
      "Temperature scaling can improve calibration post-hoc.")


---
## Section 6: Model 4 — HOG + SVM (Traditional ML Baseline)

**Histogram of Oriented Gradients (HOG)** is a hand-crafted feature descriptor
that captures the distribution of local edge directions.  It was dominant for
pedestrian detection before deep learning (Dalal & Triggs, CVPR 2005).

**Pipeline:**
1. **Positive samples:** Crop and resize person bounding boxes → extract HOG
2. **Negative samples:** Sample random background patches (no person overlap)
3. **Train LinearSVC** on HOG features + StandardScaler
4. **Sliding window detection:** Stride across test images at multiple scales
5. **Non-Maximum Suppression (NMS):** Merge overlapping boxes

The HOG+SVM model serves as our traditional ML baseline, showing quantitatively
how much deep learning has improved over classical approaches.


In [ ]:
# ─── 6.1  HOG feature extraction parameters ──────────────────────────────────
# These parameters come from the original Dalal & Triggs paper with minor
# adjustments for the person detection task.

HOG_PARAMS = {
    "orientations"       : 9,     # number of gradient orientation bins
    "pixels_per_cell"    : (8, 8),# cell size in pixels
    "cells_per_block"    : (2, 2),# block normalisation window
    "block_norm"         : "L2-Hys",
    "transform_sqrt"     : True,  # gamma correction before HOG
    "feature_vector"     : True,
}
PATCH_SIZE  = (128, 64)   # (height, width) — standard pedestrian detection size
MIN_BOX_PX  = 32          # minimum side length to accept a bounding box crop


def extract_hog(image: np.ndarray, patch_size=PATCH_SIZE) -> np.ndarray:
    """Resize image to patch_size and extract HOG features."""
    # skimage expects (H, W, C) for colour; we convert to grey for efficiency
    resized = sk_resize(image, patch_size, anti_aliasing=True)
    if resized.ndim == 3:
        # Luminance-weighted greyscale conversion
        grey = 0.2126*resized[..., 0] + 0.7152*resized[..., 1] + 0.0722*resized[..., 2]
    else:
        grey = resized
    feat = hog(grey, **HOG_PARAMS)
    return feat


# Compute feature vector length (needed to pre-allocate arrays)
dummy = np.zeros((*PATCH_SIZE, 3))
HOG_FEAT_DIM = len(extract_hog(dummy))
print(f"✅ HOG feature dimension: {HOG_FEAT_DIM}")


In [ ]:
# ─── 6.2  Build positive and negative sample datasets ────────────────────────
# Positive = person crop from ground-truth box
# Negative = random patch from image with IoU < 0.3 with any person box

def iou_xywh(boxA_xywh, boxB_xywh):
    """Compute IoU between two boxes in [x, y, w, h] format."""
    ax, ay, aw, ah = boxA_xywh
    bx, by, bw, bh = boxB_xywh
    ix1 = max(ax, bx);     iy1 = max(ay, by)
    ix2 = min(ax+aw, bx+bw); iy2 = min(ay+ah, by+bh)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    union = aw*ah + bw*bh - inter
    return inter / max(union, 1e-6)


def build_hog_dataset(img_ids, coco_api, img_dir,
                      max_pos=4000, max_neg=4000,
                      negs_per_image=3, rng=None):
    """
    Extract HOG features for positive (person) and negative (background) patches.

    Returns:
        X : np.ndarray [N, HOG_FEAT_DIM]
        y : np.ndarray [N] — 1=person, 0=background
    """
    if rng is None:
        rng = np.random.default_rng(SEED)

    pos_feats, neg_feats = [], []

    for img_id in tqdm(img_ids, desc="  Building HOG dataset", leave=False):
        if len(pos_feats) >= max_pos and len(neg_feats) >= max_neg:
            break

        img_info = coco_api.loadImgs(img_id)[0]
        img_path = img_dir / img_info["file_name"]
        if not img_path.exists():
            continue

        img_np = np.array(Image.open(img_path).convert("RGB"))
        H, W   = img_np.shape[:2]

        ann_ids = coco_api.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        anns    = coco_api.loadAnns(ann_ids)
        person_boxes = [a["bbox"] for a in anns if a["area"] >= MIN_AREA]

        # ── Positive samples ─────────────────────────────────────────────────
        if len(pos_feats) < max_pos:
            for bbox in person_boxes:
                x, y, w, h = [int(v) for v in bbox]
                if w < MIN_BOX_PX or h < MIN_BOX_PX:
                    continue
                crop = img_np[max(0,y):y+h, max(0,x):x+w]
                if crop.size == 0:
                    continue
                pos_feats.append(extract_hog(crop))

        # ── Negative samples (random background patches) ─────────────────────
        if len(neg_feats) < max_neg:
            for _ in range(negs_per_image * 5):   # try more, keep non-overlapping
                if len(neg_feats) >= max_neg:
                    break
                nw = rng.integers(PATCH_SIZE[1], min(W, 256))
                nh = rng.integers(PATCH_SIZE[0], min(H, 256))
                nx = rng.integers(0, max(1, W - nw))
                ny = rng.integers(0, max(1, H - nh))
                neg_bbox = [nx, ny, nw, nh]

                # Reject if overlaps a person box
                overlaps = any(iou_xywh(neg_bbox, pb) > 0.3 for pb in person_boxes)
                if not overlaps:
                    crop = img_np[ny:ny+nh, nx:nx+nw]
                    if crop.size > 0:
                        neg_feats.append(extract_hog(crop))

    # Balance classes by truncating the majority class
    n = min(len(pos_feats), len(neg_feats), max_pos, max_neg)
    print(f"   Positives: {len(pos_feats):,} → using {n:,}")
    print(f"   Negatives: {len(neg_feats):,} → using {n:,}")

    X = np.vstack([pos_feats[:n], neg_feats[:n]])
    y = np.concatenate([np.ones(n), np.zeros(n)])

    # Shuffle
    idx = rng.permutation(len(X))
    return X[idx], y[idx]


print("=== Building HOG training dataset ===")
X_train_hog, y_train_hog = build_hog_dataset(
    train_ids[:800], coco_val, VAL_IMG_DIR,   # use 800 images for speed
    max_pos=5000, max_neg=5000)

print(f"   X shape: {X_train_hog.shape}")
print(f"   y shape: {y_train_hog.shape}, classes: {np.unique(y_train_hog)}")


In [ ]:
# ─── 6.3  Train LinearSVM pipeline ───────────────────────────────────────────
# We persist the trained pipeline with joblib so it can be loaded
# immediately on subsequent runs without re-training.

import joblib

HOG_MODEL_PATH = MODEL_DIR / "hog_svm_pipeline.joblib"

if HOG_MODEL_PATH.exists():
    print(f"✅ HOG+SVM pipeline found — loading saved model.")
    print(f"   Loading: {HOG_MODEL_PATH}")
    hog_svm_pipeline = joblib.load(HOG_MODEL_PATH)
else:
    print("=== Training HOG + LinearSVM ===")
    t0 = time.time()
    hog_svm_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svm",    LinearSVC(C=0.01, max_iter=5000, random_state=SEED, dual=True)),
    ])
    hog_svm_pipeline.fit(X_train_hog, y_train_hog)
    train_time = time.time() - t0
    # Persist weights
    joblib.dump(hog_svm_pipeline, HOG_MODEL_PATH)
    print(f"✅ SVM trained in {train_time:.1f}s  |  Saved to {HOG_MODEL_PATH}")

# Patch-level sanity check
train_preds = hog_svm_pipeline.predict(X_train_hog)
report = classification_report(y_train_hog, train_preds,
                               target_names=["background", "person"])
print("\nPatch-level classification report (training set):")
print(report)


In [ ]:
# ─── 6.4  Sliding window + NMS detection ─────────────────────────────────────
# At test time we must localise people in full images, not just classify crops.
# Strategy:
#   1. Build an image pyramid (multiple scales) — handles people at all sizes
#   2. At each scale, slide a fixed-size window across the image
#   3. Classify each window with the HOG+SVM
#   4. Apply Non-Maximum Suppression (NMS) to remove duplicate boxes

def sliding_window(image, step_size, window_size):
    """Generator: yield (x, y, window_crop) for every window position."""
    h_win, w_win = window_size
    for y in range(0, image.shape[0] - h_win + 1, step_size):
        for x in range(0, image.shape[1] - w_win + 1, step_size):
            yield x, y, image[y:y+h_win, x:x+w_win]


def image_pyramid(image, scale=1.5, min_size=(64, 64)):
    """Generator: yield progressively down-scaled versions of the image."""
    yield image
    while True:
        h = int(image.shape[0] / scale)
        w = int(image.shape[1] / scale)
        if h < min_size[0] or w < min_size[1]:
            break
        image = np.array(Image.fromarray(image).resize((w, h)))
        yield image


def nms(boxes, scores, iou_thresh=0.3):
    """
    Non-Maximum Suppression — removes redundant overlapping boxes.
    Keeps the highest-scoring box; suppresses those with IoU > threshold.

    Args:
        boxes  : np.ndarray [N, 4] in XYXY format
        scores : np.ndarray [N]
        iou_thresh : float

    Returns: indices of kept boxes
    """
    if len(boxes) == 0:
        return np.array([], dtype=int)

    x1, y1, x2, y2 = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    areas  = (x2 - x1) * (y2 - y1)
    order  = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        inter  = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        iou    = inter / (areas[i] + areas[order[1:]] - inter + 1e-6)
        order  = order[1:][iou <= iou_thresh]

    return np.array(keep, dtype=int)


def detect_persons_hog(image_np, pipeline, window_size=PATCH_SIZE,
                       step_size=16, pyramid_scale=1.5,
                       svm_thresh=-0.5, nms_thresh=0.3):
    """
    Full sliding-window + NMS detection pipeline for a single image.

    Returns:
        boxes  : np.ndarray [N, 4] XYXY boxes after NMS
        scores : np.ndarray [N] SVM decision function scores
    """
    all_boxes, all_scores = [], []
    scale_factor = 1.0

    for scaled_img in image_pyramid(image_np, scale=pyramid_scale):
        # Extract HOG for every window at this scale
        feats, locs = [], []
        for x, y, window in sliding_window(scaled_img, step_size, window_size):
            feats.append(extract_hog(window))
            locs.append((x, y))

        if not feats:
            scale_factor *= pyramid_scale
            continue

        feats_arr = np.array(feats)
        # decision_function gives SVM confidence (distance from hyperplane)
        scores_arr = pipeline.decision_function(feats_arr)

        for (x, y), score in zip(locs, scores_arr):
            if score >= svm_thresh:
                # Map box coords back to original image scale
                x1 = int(x * scale_factor)
                y1 = int(y * scale_factor)
                x2 = int((x + window_size[1]) * scale_factor)
                y2 = int((y + window_size[0]) * scale_factor)
                all_boxes.append([x1, y1, x2, y2])
                all_scores.append(score)

        scale_factor *= pyramid_scale

    if not all_boxes:
        return np.zeros((0, 4)), np.zeros(0)

    boxes  = np.array(all_boxes, dtype=float)
    scores = np.array(all_scores)

    # Apply NMS
    keep   = nms(boxes, scores, nms_thresh)
    return boxes[keep], scores[keep]


print("✅ Sliding window + NMS functions defined.")
print(f"   Window size : {PATCH_SIZE}")
print(f"   Step size   : 16px")
print(f"   Pyramid scale: 1.5x per level")


In [ ]:
# ─── 6.5  Run HOG+SVM inference on test set ──────────────────────────────────
# WARNING: Sliding window is much slower than DL models.
# We limit to MAX_TEST_IMAGES (200) which takes ~5–15 minutes on CPU.

print("=== Model 4: HOG+SVM Inference (this may take 5–15 minutes) ===")

hog_preds  = []
total_time_hog = 0.0

for img_id in tqdm(test_ids[:MAX_TEST_IMAGES], desc="  HOG+SVM"):
    img_info = coco_val.loadImgs(img_id)[0]
    img_path = VAL_IMG_DIR / img_info["file_name"]
    if not img_path.exists():
        continue

    img_np = np.array(Image.open(img_path).convert("RGB"))

    t0 = time.time()
    boxes, scores = detect_persons_hog(
        img_np, hog_svm_pipeline,
        svm_thresh=-0.5,   # lower threshold → more recall, less precision
        nms_thresh=0.3)
    total_time_hog += time.time() - t0

    hog_preds.append({
        "image_id": img_id,
        "boxes":    boxes,
        "scores":   np.clip(scores, 0, None),  # COCOeval needs non-negative scores
        "labels":   np.ones(len(boxes), dtype=int),
    })

avg_time_hog = total_time_hog / max(len(hog_preds), 1) * 1000
print(f"\n✅ HOG+SVM inference complete.")
print(f"   Images processed : {len(hog_preds):,}")
print(f"   Avg per image    : {avg_time_hog:.0f}ms")

print("\n=== Evaluating Model 4: HOG+SVM ===")
hog_stats = run_coco_eval(coco_val, hog_preds, test_ids[:MAX_TEST_IMAGES])

# Visualise
visualise_predictions(
    test_ids[:MAX_TEST_IMAGES], coco_val, VAL_IMG_DIR, hog_preds,
    title="Model 4 — HOG + SVM",
    save_path=VIZ_DIR / "06_hog_svm_predictions.png")


### 6.6 — Model 4 Explainability & Calibration (HOG + SVM)


In [ ]:
# ─── 6.6.1  HOG+SVM — Learned Weight Visualisation ──────────────────────────
# The LinearSVC's weight vector has the same dimension as the HOG feature
# vector (HOG_FEAT_DIM).  We can reshape it back to a spatial heatmap
# showing which parts of the 128×64 patch contribute most to the "person"
# decision — this is the direct analogue of GradCAM for SVMs.

print("=== Model 4: HOG + SVM — Explainability ===")

svm_clf   = hog_svm_pipeline.named_steps["svm"]
scaler    = hog_svm_pipeline.named_steps["scaler"]

# Weight vector in original feature space (before scaling)
w = svm_clf.coef_[0]                # shape: (HOG_FEAT_DIM,)
w_scaled = w / (scaler.scale_ + 1e-8)   # undo StandardScaler to get interpretable weights

# Reconstruct spatial importance from HOG cell structure
orientations    = HOG_PARAMS["orientations"]
px_per_cell     = HOG_PARAMS["pixels_per_cell"][0]
cells_per_block = HOG_PARAMS["cells_per_block"][0]
n_cells_y = PATCH_SIZE[0] // px_per_cell
n_cells_x = PATCH_SIZE[1] // px_per_cell
n_blocks_y = n_cells_y - cells_per_block + 1
n_blocks_x = n_cells_x - cells_per_block + 1

# Average weight magnitude per cell (approximate spatial map)
cell_importance = np.zeros((n_cells_y, n_cells_x))
feat_idx = 0
for by in range(n_blocks_y):
    for bx in range(n_blocks_x):
        for cy in range(cells_per_block):
            for cx in range(cells_per_block):
                cell_y = by + cy; cell_x = bx + cx
                block_feat = w_scaled[feat_idx:feat_idx + orientations]
                cell_importance[cell_y, cell_x] += np.abs(block_feat).mean()
                feat_idx += orientations

# Normalise importance map
ci_norm = (cell_importance - cell_importance.min()) / (cell_importance.max() - cell_importance.min() + 1e-8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Model 4 — HOG + SVM: Learned Weight Visualisation",
             fontsize=12, fontweight="bold")

# Cell importance heatmap
im = axes[0].imshow(ci_norm, cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_title(f"SVM Weight Map ({n_cells_y}×{n_cells_x} HOG cells)\n"
                  "Green = positive (person), Red = negative (bg)")
axes[0].set_xlabel("Cell X"); axes[0].set_ylabel("Cell Y")

# Overlaid on the mean positive patch
if "mean_patch" in dir() and mean_patch is not None:
    ci_upscaled = np.array(
        Image.fromarray((ci_norm*255).astype(np.uint8)).resize(
            (PATCH_SIZE[1], PATCH_SIZE[0]), Image.NEAREST)) / 255.0
    axes[1].imshow(mean_patch)
    axes[1].imshow(ci_upscaled, cmap="jet", alpha=0.55)
    axes[1].set_title("Weight Map on Mean Patch")
    axes[1].axis("off")
else:
    axes[1].text(0.5, 0.5, "Mean patch not available\n(EDA skipped)",
                 ha="center", va="center", transform=axes[1].transAxes)
    axes[1].axis("off")

# Weight distribution
axes[2].hist(w, bins=60, color="#8172B2", edgecolor="white", lw=0.5)
axes[2].axvline(0, color="black", lw=1.5, label="Decision boundary")
axes[2].set_title(f"SVM Weight Distribution\n(dim={len(w):,})")
axes[2].set_xlabel("Weight value"); axes[2].legend()

plt.tight_layout()
plt.savefig(VIZ_DIR / "EXPLAIN_04_hog_svm_weights.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ HOG+SVM explainability complete.")


In [ ]:
# ─── 6.6.2  HOG+SVM — Calibration ────────────────────────────────────────────
# LinearSVC's decision_function gives an *unbounded* distance score, not a
# probability.  We clip negative scores to 0 for COCO eval.  Here we analyse
# the score distribution and reliability.
#
# Note: For a proper probability output, we would use CalibratedClassifierCV
#       (Platt scaling or isotonic regression) in a production system.

print("=== Model 4: HOG + SVM — Calibration ===")

# Normalise HOG scores to [0,1] for reliability diagram
all_hog_scores_raw = np.concatenate([p["scores"] for p in hog_preds if len(p["scores"])])

if len(all_hog_scores_raw) > 0:
    s_min, s_max = all_hog_scores_raw.min(), all_hog_scores_raw.max()
    hog_preds_norm = []
    for p in hog_preds:
        if len(p["scores"]):
            norm_scores = (p["scores"] - s_min) / max(s_max - s_min, 1e-8)
            hog_preds_norm.append({**p, "scores": norm_scores})
        else:
            hog_preds_norm.append(p)
else:
    hog_preds_norm = hog_preds

bc_hog, ba_hog, bcount_hog, ece_hog = compute_detection_calibration(
    hog_preds_norm, coco_val, test_ids[:MAX_TEST_IMAGES], n_bins=10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model 4 — HOG + SVM: Calibration (scores normalised to [0,1])",
             fontsize=12, fontweight="bold")

plot_reliability_diagram(bc_hog, ba_hog, bcount_hog, ece_hog,
                         title="Reliability Diagram (SVM scores)", ax=axes[0])

axes[1].hist(all_hog_scores_raw, bins=50, color="#F44336", edgecolor="white", lw=0.5)
axes[1].set_title("Raw SVM Decision Function Score Distribution")
axes[1].set_xlabel("SVM score (distance from hyperplane)")
axes[1].set_ylabel("# predictions")
axes[1].axvline(0, color="black", lw=1.5, label="SVM boundary (score=0)")
axes[1].legend()

plt.tight_layout()
plt.savefig(VIZ_DIR / "CALIB_04_hog_svm.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"  ECE (HOG+SVM, normalised): {ece_hog:.4f}")
print("  ⚠ ECE is approximate — LinearSVC scores are not true probabilities.")
print("  For proper calibration, use CalibratedClassifierCV (Platt scaling).")


---
## Section 7: Unified Evaluation & Comparison

We now bring all four models together for a comprehensive comparison across
multiple dimensions:
- **Detection accuracy** (mAP@0.5, mAP@0.5:0.95)
- **Inference speed** (ms per image)
- **Precision–Recall curves**
- **Failure analysis** (missed detections, false positives)


In [ ]:
# ─── 7.1  Precision–Recall curve computation ─────────────────────────────────
# We compute PR curves by varying the confidence threshold and computing
# precision and recall at each threshold.  This gives a more complete picture
# than a single operating point.

def compute_pr_curve(predictions, gt_coco, img_ids, iou_thresh=IOU_THRESHOLD):
    """
    Compute precision and recall at multiple confidence thresholds.

    Uses a simple matching strategy:
        For each threshold t:
            - TP: predicted box with score ≥ t that matches a GT box (IoU ≥ thresh)
            - FP: predicted box with score ≥ t that does not match any GT box
            - FN: GT box not matched by any prediction with score ≥ t
    """
    # Collect all predictions with their GT status
    all_scores = []
    all_tp     = []   # 1 if this prediction is a TP

    # Build GT lookup
    gt_map = {}
    for img_id in img_ids:
        ann_ids = gt_coco.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        anns    = gt_coco.loadAnns(ann_ids)
        gt_map[img_id] = np.array([a["bbox"] for a in anns
                                   if a["area"] >= MIN_AREA], dtype=float)

    pred_map = {p["image_id"]: p for p in predictions}

    for img_id in img_ids:
        gt_boxes_xywh = gt_map.get(img_id, np.zeros((0, 4)))
        pred = pred_map.get(img_id)
        if pred is None or len(pred["boxes"]) == 0:
            continue

        pred_boxes  = pred["boxes"]    # XYXY
        pred_scores = pred["scores"]

        # Convert GT to XYXY
        if gt_boxes_xywh.shape[0] > 0:
            gt_xyxy = gt_boxes_xywh.copy()
            gt_xyxy[:, 2] += gt_xyxy[:, 0]
            gt_xyxy[:, 3] += gt_xyxy[:, 1]
        else:
            gt_xyxy = np.zeros((0, 4))

        matched_gt = set()

        # Sort predictions by score descending
        order = np.argsort(-pred_scores)
        for idx in order:
            box   = pred_boxes[idx]
            score = pred_scores[idx]
            all_scores.append(score)

            if gt_xyxy.shape[0] == 0:
                all_tp.append(0)
                continue

            # Compute IoU with all GT boxes
            x1 = np.maximum(box[0], gt_xyxy[:, 0])
            y1 = np.maximum(box[1], gt_xyxy[:, 1])
            x2 = np.minimum(box[2], gt_xyxy[:, 2])
            y2 = np.minimum(box[3], gt_xyxy[:, 3])
            inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
            area_pred = (box[2]-box[0]) * (box[3]-box[1])
            areas_gt  = (gt_xyxy[:,2]-gt_xyxy[:,0]) * (gt_xyxy[:,3]-gt_xyxy[:,1])
            union = area_pred + areas_gt - inter
            ious  = inter / np.maximum(union, 1e-6)

            # Match to best unmatched GT
            best_gt = np.argmax(ious)
            if ious[best_gt] >= iou_thresh and best_gt not in matched_gt:
                matched_gt.add(best_gt)
                all_tp.append(1)
            else:
                all_tp.append(0)

    if not all_scores:
        return np.array([0.0, 1.0]), np.array([1.0, 0.0])

    all_scores = np.array(all_scores)
    all_tp     = np.array(all_tp)

    # Total GT count across all images
    n_gt = sum(len(gt_map.get(i, [])) for i in img_ids)

    # Sort by descending score
    order      = np.argsort(-all_scores)
    tp_sorted  = all_tp[order]
    cum_tp     = np.cumsum(tp_sorted)
    cum_fp     = np.cumsum(1 - tp_sorted)

    precision = cum_tp / (cum_tp + cum_fp + 1e-8)
    recall    = cum_tp / max(n_gt, 1)

    # Append boundary points
    precision = np.concatenate([[1.0], precision])
    recall    = np.concatenate([[0.0], recall])

    return recall, precision


# Compute PR curves for all models (use shared test_ids subset)
eval_ids = test_ids[:MAX_TEST_IMAGES]   # use same subset for fair comparison

print("Computing PR curves …")
pr_curves = {}
pr_curves["YOLOv8 Baseline"]   = compute_pr_curve(yolo_base_preds, coco_val, eval_ids)
pr_curves["YOLOv8 Fine-Tuned"] = compute_pr_curve(yolo_ft_preds,   coco_val, eval_ids)
pr_curves["Faster R-CNN"]      = compute_pr_curve(frcnn_preds,      coco_val, eval_ids)
pr_curves["HOG + SVM"]         = compute_pr_curve(hog_preds,        coco_val, eval_ids)
print("✅ PR curves computed.")


In [ ]:
# ─── 7.2  Plot Precision–Recall curves ───────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 6))
colours = {"YOLOv8 Baseline": "steelblue", "YOLOv8 Fine-Tuned": "darkorange",
           "Faster R-CNN": "green", "HOG + SVM": "red"}
styles  = {"YOLOv8 Baseline": "-",  "YOLOv8 Fine-Tuned": "--",
           "Faster R-CNN": "-.",    "HOG + SVM": ":"}

for name, (recall, precision) in pr_curves.items():
    # Compute AUC (approximate average precision)
    auc = trapezoid(precision, recall)
    ax.plot(recall, precision,
            label=f"{name}  (AP≈{auc:.3f})",
            color=colours[name], linestyle=styles[name], linewidth=2)

ax.set_xlabel("Recall",    fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision–Recall Curves — Human Person Detection"
             "All models evaluated on the same held-out test set (IoU@0.5)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10, loc="upper right")
ax.set_xlim([0, 1.05]); ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3)
ax.fill_between([0, 1], [0, 0], alpha=0.0)  # invisible, just for layout

plt.tight_layout()
plt.savefig(VIZ_DIR / "07_precision_recall_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ PR curve plot saved.")

In [ ]:
# ─── 7.3  Summary comparison table ───────────────────────────────────────────

def extract_summary(stats_dict, avg_time_ms, model_type="DL"):
    """Build a flat summary dict from COCOeval stats."""
    return {
        "mAP@0.5":       round(stats_dict.get("mAP_50",    0.0), 4),
        "mAP@0.5:0.95":  round(stats_dict.get("mAP_50_95", 0.0), 4),
        "AR@100":         round(stats_dict.get("AR_100",    0.0), 4),
        "Avg time (ms)":  round(avg_time_ms, 1),
        "Type":           model_type,
    }

results_table = {
    "YOLOv8 Baseline":   extract_summary(yolo_base_stats, avg_time_base,  "DL"),
    "YOLOv8 Fine-Tuned": extract_summary(yolo_ft_stats,   avg_time_ft,    "DL"),
    "Faster R-CNN":      extract_summary(frcnn_stats,     avg_time_frcnn, "DL"),
    "HOG + SVM":         extract_summary(hog_stats,       avg_time_hog,   "TML"),
}

# Pretty-print table
header = f"{'Model':<22} {'mAP@0.5':>10} {'mAP@0.5:0.95':>14} {'AR@100':>8} {'ms/img':>8} {'Type':>5}"
print("=" * len(header))
print("  FINAL MODEL COMPARISON — Human Person Detection")
print("=" * len(header))
print(header)
print("-" * len(header))
for model_name, row in results_table.items():
    print(f"{model_name:<22} {row['mAP@0.5']:>10.4f} {row['mAP@0.5:0.95']:>14.4f} "
          f"{row['AR@100']:>8.4f} {row['Avg time (ms)']:>8.1f} {row['Type']:>5}")
print("=" * len(header))

# Save results
with open(RESULTS_DIR / "comparison_table.json", "w") as f:
    json.dump(results_table, f, indent=2)
print(f"\n✅ Results saved to {RESULTS_DIR / 'comparison_table.json'}")


In [ ]:
# ─── 7.4  Bar chart comparison ───────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Model Comparison — Human Person Detection on COCO",
             fontsize=13, fontweight="bold")

models      = list(results_table.keys())
map50       = [results_table[m]["mAP@0.5"]      for m in models]
map50_95    = [results_table[m]["mAP@0.5:0.95"] for m in models]
speed       = [results_table[m]["Avg time (ms)"]for m in models]

bar_colours = ["#2196F3", "#FF9800", "#4CAF50", "#F44336"]

# mAP@0.5
axes[0].bar(models, map50, color=bar_colours)
axes[0].set_title("mAP @ IoU=0.5", fontsize=11)
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel("mAP"); axes[0].tick_params(axis="x", rotation=25)
for i, v in enumerate(map50):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

# mAP@0.5:0.95
axes[1].bar(models, map50_95, color=bar_colours)
axes[1].set_title("mAP @ IoU=0.5:0.95", fontsize=11)
axes[1].set_ylim(0, 1.0)
axes[1].set_ylabel("mAP"); axes[1].tick_params(axis="x", rotation=25)
for i, v in enumerate(map50_95):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

# Inference speed (log scale because HOG+SVM is much slower)
axes[2].bar(models, speed, color=bar_colours)
axes[2].set_title("Avg Inference Time (ms/image)", fontsize=11)
axes[2].set_ylabel("ms"); axes[2].tick_params(axis="x", rotation=25)
axes[2].set_yscale("log")
for i, v in enumerate(speed):
    axes[2].text(i, v * 1.1, f"{v:.0f}ms", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(VIZ_DIR / "08_model_comparison_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Comparison bar chart saved.")


### Section 7.5 — Combined Calibration & Explainability Summary


In [ ]:
# ─── 7.5  Cross-model Calibration Comparison ──────────────────────────────────
# Overlay all four reliability diagrams for direct comparison.
# Also summarise ECE values.

print("=== Cross-Model Calibration Comparison ===")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Reliability Diagrams — All Models\n"
             "Closer to the dashed diagonal = better calibration",
             fontsize=13, fontweight="bold")

models_calib = [
    ("YOLOv8 Baseline",   bc_base,  ba_base,  bcount_base,  ece_base,  "#2196F3"),
    ("YOLOv8 Fine-Tuned", bc_ft,    ba_ft,    bcount_ft,    ece_ft,    "#FF9800"),
    ("Faster R-CNN",      bc_frcnn, ba_frcnn, bcount_frcnn, ece_frcnn, "#4CAF50"),
    ("HOG + SVM",         bc_hog,   ba_hog,   bcount_hog,   ece_hog,   "#F44336"),
]

for ax, (name, bc, ba, bcnt, ece, col) in zip(axes.flat, models_calib):
    plot_reliability_diagram(bc, ba, bcnt, ece, title=name, ax=ax)

plt.tight_layout()
plt.savefig(VIZ_DIR / "CALIB_05_all_models_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

# ECE Summary table
print("\n" + "=" * 50)
print("  Calibration Summary (ECE — lower is better)")
print("=" * 50)
for name, _, _, _, ece, _ in models_calib:
    bar = "█" * int(ece * 30)
    print(f"  {name:<22} ECE={ece:.4f}  {bar}")
print("=" * 50)
print("  * HOG+SVM ECE is approximate (unnormalised scores).")
print("✅ Calibration comparison complete.")


---
## Section 8: Failure Analysis & Visual Comparison

Understanding *where* models fail is as important as overall metrics.
We categorise failures into:
- **False Negatives (missed detections):** A person is present but not detected
- **False Positives:** A detection is made but no person is there (hallucination)

We display side-by-side comparisons of the same images across all models.


In [ ]:
# ─── 8.1  Identify failure cases ─────────────────────────────────────────────

def find_failure_cases(predictions, gt_coco, img_ids,
                       iou_thresh=IOU_THRESHOLD, conf_thresh=CONF_THRESHOLD):
    """
    For each image, determine:
        - missed: images with GT people that no prediction covers (FN)
        - false_pos: images with predictions that hit no GT box (FP)
    Returns two lists of image_ids.
    """
    pred_map = {p["image_id"]: p for p in predictions}
    missed_ids     = []
    false_pos_ids  = []

    for img_id in img_ids:
        ann_ids = gt_coco.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
        anns    = [a for a in gt_coco.loadAnns(ann_ids) if a["area"] >= MIN_AREA]
        gt_boxes_xywh = np.array([a["bbox"] for a in anns], dtype=float)

        pred = pred_map.get(img_id)
        pred_boxes = pred["boxes"]  if (pred is not None and len(pred["boxes"]) > 0) else np.zeros((0,4))

        # Convert GT to XYXY
        if gt_boxes_xywh.shape[0] > 0:
            gt_xyxy = gt_boxes_xywh.copy()
            gt_xyxy[:, 2] += gt_xyxy[:, 0]
            gt_xyxy[:, 3] += gt_xyxy[:, 1]
        else:
            gt_xyxy = np.zeros((0, 4))

        # Check for missed detections
        if gt_xyxy.shape[0] > 0 and pred_boxes.shape[0] == 0:
            missed_ids.append(img_id)
        elif gt_xyxy.shape[0] > 0 and pred_boxes.shape[0] > 0:
            # Check if any GT is uncovered
            for gt_box in gt_xyxy:
                x1 = np.maximum(gt_box[0], pred_boxes[:, 0])
                y1 = np.maximum(gt_box[1], pred_boxes[:, 1])
                x2 = np.minimum(gt_box[2], pred_boxes[:, 2])
                y2 = np.minimum(gt_box[3], pred_boxes[:, 3])
                inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
                area_gt = (gt_box[2]-gt_box[0]) * (gt_box[3]-gt_box[1])
                areas_pred = (pred_boxes[:,2]-pred_boxes[:,0])*(pred_boxes[:,3]-pred_boxes[:,1])
                union = area_gt + areas_pred - inter
                ious  = inter / np.maximum(union, 1e-6)
                if ious.max() < iou_thresh:
                    missed_ids.append(img_id)
                    break

        # Check for false positives
        if pred_boxes.shape[0] > 0:
            for pred_box in pred_boxes:
                if gt_xyxy.shape[0] == 0:
                    false_pos_ids.append(img_id)
                    break
                x1 = np.maximum(pred_box[0], gt_xyxy[:, 0])
                y1 = np.maximum(pred_box[1], gt_xyxy[:, 1])
                x2 = np.minimum(pred_box[2], gt_xyxy[:, 2])
                y2 = np.minimum(pred_box[3], gt_xyxy[:, 3])
                inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
                area_pred = (pred_box[2]-pred_box[0]) * (pred_box[3]-pred_box[1])
                areas_gt  = (gt_xyxy[:,2]-gt_xyxy[:,0]) * (gt_xyxy[:,3]-gt_xyxy[:,1])
                union = area_pred + areas_gt - inter
                ious  = inter / np.maximum(union, 1e-6)
                if ious.max() < iou_thresh:
                    false_pos_ids.append(img_id)
                    break

    return list(set(missed_ids)), list(set(false_pos_ids))


print("Analysing failure cases for YOLOv8 Fine-Tuned (best DL model) …")
missed_ft, fp_ft = find_failure_cases(yolo_ft_preds, coco_val, eval_ids)
print(f"  Missed detections : {len(missed_ft)}")
print(f"  False positives   : {len(fp_ft)}")


In [ ]:
# ─── 8.2  Visualise failure cases ────────────────────────────────────────────

def visualise_failures(img_ids, coco_api, img_dir, predictions,
                       title, save_path, n_show=6):
    """Show images where the model failed."""
    pred_map = {p["image_id"]: p for p in predictions}
    ids = [i for i in img_ids if i in pred_map][:n_show]

    if not ids:
        print(f"   No failure examples to show for: {title}")
        return

    n = min(len(ids), n_show)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    fig.suptitle(title + "\n🟩 GT  🟥 Prediction", fontsize=11, fontweight="bold")
    for ax, img_id in zip(axes.flat, ids):
        draw_coco_boxes(ax, img_id, coco_api, img_dir)
        pred = pred_map.get(img_id)
        if pred:
            for (x1,y1,x2,y2) in pred["boxes"]:
                ax.add_patch(mpatches.Rectangle(
                    (x1,y1), x2-x1, y2-y1, linewidth=2, edgecolor="red", facecolor="none"))
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()


visualise_failures(
    missed_ft, coco_val, VAL_IMG_DIR, yolo_ft_preds,
    "YOLOv8 Fine-Tuned — Missed Detections (False Negatives)",
    VIZ_DIR / "09_missed_detections.png")

visualise_failures(
    fp_ft, coco_val, VAL_IMG_DIR, yolo_ft_preds,
    "YOLOv8 Fine-Tuned — False Positives",
    VIZ_DIR / "10_false_positives.png")


In [ ]:
# ─── 8.3  Side-by-side four-model comparison ─────────────────────────────────
# Show the same 3 images as predicted by all four models simultaneously.

def four_model_comparison(img_ids, coco_api, img_dir,
                          preds_dict, n_images=3, save_path=None):
    """
    Grid: each row = one image, each column = one model.
    Green = GT, Red = prediction.
    """
    model_names = list(preds_dict.keys())
    n_models    = len(model_names)
    ids_to_show = img_ids[:n_images]

    fig, axes = plt.subplots(n_images, n_models,
                             figsize=(5 * n_models, 4 * n_images))
    if n_images == 1:
        axes = axes[np.newaxis, :]

    for col, model_name in enumerate(model_names):
        axes[0, col].set_title(model_name, fontsize=10, fontweight="bold", pad=8)
        pred_map = {p["image_id"]: p for p in preds_dict[model_name]}

        for row, img_id in enumerate(ids_to_show):
            ax = axes[row, col]
            img_info = coco_api.loadImgs(img_id)[0]
            img_path = img_dir / img_info["file_name"]
            if not img_path.exists():
                ax.axis("off"); continue

            ax.imshow(np.array(Image.open(img_path).convert("RGB")))

            # GT boxes
            ann_ids = coco_api.getAnnIds(imgIds=img_id, catIds=[PERSON_CAT_ID])
            for ann in coco_api.loadAnns(ann_ids):
                x,y,w,h = ann["bbox"]
                ax.add_patch(mpatches.Rectangle((x,y),w,h,
                    linewidth=2, edgecolor="lime", facecolor="none"))

            # Predicted boxes
            pred = pred_map.get(img_id)
            if pred and len(pred["boxes"]) > 0:
                for (x1,y1,x2,y2) in pred["boxes"]:
                    ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,
                        linewidth=2, edgecolor="red", facecolor="none"))
            ax.axis("off")

    fig.suptitle("Four-Model Side-by-Side Comparison\n🟩 Ground Truth  🟥 Prediction",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()


preds_dict = {
    "YOLOv8\nBaseline":   yolo_base_preds,
    "YOLOv8\nFine-Tuned": yolo_ft_preds,
    "Faster\nR-CNN":      frcnn_preds,
    "HOG +\nSVM":         hog_preds,
}

four_model_comparison(
    eval_ids, coco_val, VAL_IMG_DIR, preds_dict,
    n_images=3,
    save_path=VIZ_DIR / "11_four_model_comparison.png")

print("✅ All visualisations complete.")


---
## Section 9: Final Summary & Key Findings

This section prints a consolidated summary of all experimental results and
highlights the key insights from the comparison.


In [ ]:
# ─── 9.1  Print complete results summary ─────────────────────────────────────

print("=" * 70)
print("  EXPERIMENT SUMMARY — Human Person Detection on COCO 2017")
print("=" * 70)
print(f"  Dataset    : COCO 2017 val2017 (person class focus)")
print(f"  Train set  : {len(train_ids):,} images")
print(f"  Val set    : {len(val_ids):,}  images")
print(f"  Test set   : {len(test_ids):,}   images (evaluation subset: {len(eval_ids)})")
print(f"  Seed       : {SEED}")
print()

print(f"{'Model':<22} {'mAP@0.5':>9} {'mAP@.5:.95':>11} {'AR@100':>7} {'ms/img':>8}")
print("-" * 60)
for m, row in results_table.items():
    print(f"{m:<22} {row['mAP@0.5']:>9.4f} {row['mAP@0.5:0.95']:>11.4f} "
          f"{row['AR@100']:>7.4f} {row['Avg time (ms)']:>8.1f}")

print()
print("KEY FINDINGS:")
print("-" * 60)

# Determine best DL model
dl_models = {k: v for k, v in results_table.items() if v["Type"] == "DL"}
best_dl    = max(dl_models, key=lambda k: dl_models[k]["mAP@0.5"])
hog_map    = results_table["HOG + SVM"]["mAP@0.5"]
best_dl_map= results_table[best_dl]["mAP@0.5"]
base_map   = results_table["YOLOv8 Baseline"]["mAP@0.5"]
ft_map     = results_table["YOLOv8 Fine-Tuned"]["mAP@0.5"]

print(f"  1. Best model       : {best_dl} (mAP@0.5 = {best_dl_map:.4f})")
print(f"  2. Fine-tuning gain : +{(ft_map - base_map)*100:.1f}% mAP@0.5 "
      f"(Baseline {base_map:.4f} → Fine-tuned {ft_map:.4f})")
print(f"  3. DL vs ML gap     : +{(best_dl_map - hog_map)*100:.1f}% mAP@0.5 "
      f"over HOG+SVM ({hog_map:.4f})")
print(f"  4. Speed comparison : YOLOv8 is {results_table['HOG + SVM']['Avg time (ms)'] / results_table['YOLOv8 Baseline']['Avg time (ms)']:.0f}× "
      f"faster than HOG+SVM")
print()
print(f"  Saved visualisations : {VIZ_DIR.resolve()}")
print(f"  Saved results JSON   : {RESULTS_DIR / 'comparison_table.json'}")
print("=" * 70)
print("✅ Notebook complete.  All sections run successfully.")


In [ ]:
# ─── 9.2  Generate final visualisation montage ───────────────────────────────
# Collect all saved plots and show them in a final grid summary.

import matplotlib.image as mpimg

plot_files = sorted(VIZ_DIR.glob("*.png"))
n = len(plot_files)
if n > 0:
    cols = 3
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(18, 6 * rows))
    axes = axes.flat

    for ax, pf in zip(axes, plot_files):
        try:
            img = mpimg.imread(str(pf))
            ax.imshow(img)
            ax.set_title(pf.name, fontsize=7)
        except Exception:
            pass
        ax.axis("off")

    for ax in list(axes)[n:]:
        ax.axis("off")

    fig.suptitle("All Saved Visualisations", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "montage.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"✅ Montage saved to {RESULTS_DIR / 'montage.png'}")
